# Trust-Drift Fusion (TDF) — Full Evaluation Suite (TON_IoT)
Real TON_IoT network traffic data (loaded from the project's GitHub repo, with a local-file fallback). Covers: baseline comparison (5 models, 8 metrics),
ablation study, context-aware (per-regime) fusion, cost-sensitive thresholds, SHAP
explainability, drift-adaptive retraining, and adversarial robustness testing.

## How to read this notebook (guide added in the commented copy)

**What the project does.** Trust-Drift Fusion (TDF) is a network-intrusion detector. For every network flow it computes three signals and fuses them into one risk score:

| Signal | Meaning | Built from |
|---|---|---|
| **AS** (Anomaly Score) | How unusual is this flow? | Isolation Forest (unsupervised) |
| **RDS** (Resource-Drift Score) | How far has behaviour drifted from the *Normal* baseline? | Mean absolute z-score vs. Normal-only statistics |
| **RF** (Random-Forest probability) | How likely is this flow an attack? | Random Forest (supervised) |

**HT (Hybrid Threat score)** = `w_as*AS + w_rds*RDS + w_rf*RF`, with the three weights summing to 1. The weights and the decision threshold are learned on a *validation* split. **Trust Score** = `100 - 100*HT` (high = trustworthy), mapped to bands Trusted / Monitored / Elevated / Critical.

**Pipeline map**

| Step | Section | Purpose |
|---|---|---|
| 0 | Config + imports | Settings, seed, dependency check |
| 1 | Load + preprocess | Read CSV (GitHub first, local fallback), clean schema |
| 2 | Models | 4 baselines + TDF |
| 3 | Baseline comparison | Split, encode, fit, evaluate on held-out test, Table 1 and figures |
| 4 | Ablation | Contribution of each signal |
| 5 | Context-aware fusion | Per-regime weights/thresholds |
| 6 | Cost-sensitive thresholds | Missed attack costs more than a false alarm |
| 7 | Explainability | SHAP-based per-flow explanations |
| 8 | Drift-adaptive retraining | Synthetic stream, retrain when drift persists |
| 9 | Adversarial stress test | Can an attacker evade the RF / the fusion? |
| 10 | Feature-group diagnostic | Do high scores depend on ports / DNS / HTTP / SSL fields? |
| 11 | Final certification | Integrity gate and run manifest |
| 12 | Downloads | Collect the result files |

**Golden rule used everywhere (leakage control).** The test set is touched exactly once, at the end, only to *report* metrics. Anything that is *learned* (scaler, one-hot vocabulary, weights, thresholds, regimes) comes from training data only. Validation splits are carved out of the training data.

**Run order.** Run top to bottom in a fresh kernel. The configuration cell must run before the dependency cell because the dependency cell uses `RANDOM_STATE`.


### What this cell does: configuration
Single place for run-wide settings: execution mode (`real` vs `self_test`), the random seed, split sizes, and where to find the data (GitHub URL, environment variable, local paths). Think of it as the control panel of the experiment.

In [ ]:
# ==============================================================================
# CONFIGURATION CELL. Run this first: it defines the run-wide settings that later cells
# read. Change settings here, not inside later cells.
# ==============================================================================
# Reproducibility configuration.
# "real" runs the publication experiment on the original ToN_IoT CSV; "self_test" runs a small
# synthetic dataset so the code can be checked without any data.

# 'real' = use the actual TON_IoT CSV (the publication run). 'self_test' = a small synthetic
# dataset that only proves the code runs; its numbers are NOT scientific results.
EXECUTION_MODE = "real"  # "self_test" or "real"
# Seed for every random operation, so a re-run reproduces the same numbers.
RANDOM_STATE = 42
# Run parameters. Later cells read these: train/test split, validation split, fusion-weight
# grid step, number of trees and Random Forest depth (None = unlimited).
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.25
WEIGHT_STEP = 0.05
N_ESTIMATORS = 200
RF_MAX_DEPTH = None

# Expected real-data filename. You may override it with an environment variable:
# %env TON_IOT_CSV=/absolute/path/to/train_test_network.csv
# %env TON_IOT_CSV=/content/train_test_network.csv
# Name of the environment variable that can hold an absolute path to the CSV (local
# fallback, option 1).
DATA_ENV_VAR = "TON_IOT_CSV"
# Relative paths tried in order when the env var is not set (local fallback, option 2).
DATA_CANDIDATES = [
    "train_test_network.csv",
    "Train_Test_Network.csv",
    "data/train_test_network.csv",
    "TON_IoT/train_test_network.csv",
]

# PRIMARY data source: the CSV in your GitHub repo. A normal github.com/.../blob/... link is
# converted to the raw-file URL by the loader. Set to "" to skip GitHub and use only local
# files.
GITHUB_CSV_URL = "https://github.com/l0keshkum4r/TDF/blob/main/train_test_network.csv"
# Private repo only: set a personal access token in this environment variable.
GITHUB_TOKEN_ENV_VAR = "GITHUB_TOKEN"

# Download option: True = also save the CSV downloaded from GitHub as ./train_test_network.csv,
# so later runs (and offline use) can read the local copy. False = keep it in memory only.
SAVE_LOCAL_COPY = False

# Fail-fast sanity checks on the settings above.
assert EXECUTION_MODE in {"real", "self_test"}
assert 0 < TEST_SIZE < 1
assert 0 < VALIDATION_SIZE < 1
assert 0 < WEIGHT_STEP <= 0.5
print("Mode:", EXECUTION_MODE)
print("Random state:", RANDOM_STATE)


### What this cell does: dependency check and imports
Confirms that numpy, pandas, scikit-learn, matplotlib and shap are installed and imports everything the notebook needs. It stops with a readable error if something is missing instead of failing halfway through a long run.

In [ ]:
# ==============================================================================
# DEPENDENCY CHECK + ALL IMPORTS. Verifies the required packages are installed and imports
# everything the rest of the notebook uses. It deliberately does not pip-install anything,
# so the environment stays explicit and reproducible.
# ==============================================================================
# Dependency check. The notebook intentionally does not auto-install packages.
# Install once in the environment if needed:
#   python -m pip install numpy pandas scikit-learn matplotlib shap

import importlib.util
# Module names exactly as Python imports them (scikit-learn is imported as 'sklearn').
required_modules = ["numpy", "pandas", "sklearn", "matplotlib", "shap"]
# find_spec() returns None for a package that is not installed.
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
# Stop immediately with a clear message instead of failing later in some random cell.
if missing:
    raise ImportError(
        "Missing required Python packages: " + ", ".join(missing) +
        ". Install them with: python -m pip install numpy pandas scikit-learn matplotlib shap"
    )

# Standard-library imports (io / re / urllib.request: GitHub loader; hashlib / datetime: final audit).
import io
import re
import urllib.request
import os
import json
import hashlib
import datetime
import warnings
from pathlib import Path

# Core data-science libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn pieces: KMeans (regime discovery), IsolationForest (unsupervised anomaly detector),
# RandomForestClassifier (supervised), LogisticRegression and GaussianNB (baselines), metrics,
# train/test splitting and feature scaling.
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

# Hide noisy sklearn UserWarnings so the output stays readable.
warnings.filterwarnings("ignore", category=UserWarning)
# Seed NumPy's global random generator. Works because RANDOM_STATE was defined in the
# configuration cell above, so run that cell first.
np.random.seed(RANDOM_STATE)
import sklearn
# Print versions so a run can be audited and reproduced later.
print("Core package versions:")
print({"numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__})
print("All required core packages are available.")


## 1. Load real dataset (TON_IoT)

### Obtaining the data

1. **Default: load directly from GitHub.** Section 1 reads the CSV from the repo set in `GITHUB_CSV_URL` in the configuration cell (currently <https://github.com/l0keshkum4r/TDF/blob/main/train_test_network.csv>). A public repo needs no account or token, and the normal `github.com/.../blob/...` link is converted to the raw-file link automatically.
2. **Download option.** Either set `SAVE_LOCAL_COPY = True` in the configuration cell, which saves the downloaded file as `./train_test_network.csv` while loading, or download it yourself:
   - Browser: open the file page on GitHub and click **Download raw file**.
   - Command line: `curl -L -o train_test_network.csv https://raw.githubusercontent.com/l0keshkum4r/TDF/main/train_test_network.csv`
   - Whole repo: `git clone https://github.com/l0keshkum4r/TDF.git`
3. **Use a local copy (offline).** Put the file at `./train_test_network.csv` (or `data/`, `TON_IoT/`), or point the `TON_IOT_CSV` environment variable at it, and set `GITHUB_CSV_URL = ""` to skip GitHub. If the GitHub download fails, the notebook automatically falls back to these local locations.


### What this cell does: load the dataset
Reads the TON_IoT CSV into a DataFrame `df`.
1. If `GITHUB_CSV_URL` is set, download from GitHub (a normal `github.com/.../blob/...` link is converted to the raw-file link automatically).
2. If that fails or is empty, fall back to local files (`TON_IOT_CSV` environment variable, then the candidate paths).
3. In `self_test` mode, a small synthetic dataset is generated instead.

It then prints the source, row count, column count and label balance, so every run records where its data came from.

In [ ]:
# ==============================================================================
# DATA LOADING. Loads the dataset into `df` and records its shape. Order of preference in
# 'real' mode: (1) GitHub URL, (2) local file via TON_IOT_CSV env var, (3) local candidate
# paths. It never silently substitutes a different dataset.
# ==============================================================================
# Builds a small FAKE dataset with the same column names as TON_IoT. Used only in
# 'self_test' mode to check that the whole pipeline runs end to end.
def _make_self_test_dataset(seed=42, n=1800):
    rng = np.random.default_rng(seed)
    # y = label: 1 = attack (about 35% of rows), 0 = normal.
    y = rng.binomial(1, 0.35, size=n)

    # Numeric features. Attack rows get larger bytes / packets / rates / error values so the
    # two classes are separable.
    numeric = {
        "duration": np.clip(rng.lognormal(mean=0.1 + 0.7*y, sigma=0.7, size=n), 0, 40),
        "src_bytes": np.maximum(0, rng.normal(800 + 2200*y, 350 + 500*y, size=n)),
        "dst_bytes": np.maximum(0, rng.normal(900 + 1600*y, 300 + 450*y, size=n)),
        "src_pkts": np.maximum(1, rng.normal(8 + 10*y, 2 + 5*y, size=n)),
        "dst_pkts": np.maximum(1, rng.normal(7 + 8*y, 2 + 4*y, size=n)),
        "conn_rate": np.maximum(0, rng.normal(3 + 12*y, 1 + 5*y, size=n)),
        "error_rate": np.clip(rng.normal(0.02 + 0.20*y, 0.01 + 0.05*y, size=n), 0, 1),
    }
    df = pd.DataFrame(numeric)
    # Categorical features, also correlated with the label. These exercise the one-hot
    # encoding path used for the real data.
    df["proto"] = np.where(rng.random(n) < 0.80 - 0.25*y, "tcp", "udp")
    df["service"] = np.where(rng.random(n) < 0.65 - 0.20*y, "http", "unknown")
    df["conn_state"] = np.where(y == 1, "REJ", np.where(rng.random(n) < 0.7, "SF", "S0"))
    # Identifier columns (IPs, timestamp, attack type) are included on purpose so the 'drop
    # identifiers' step in the next cell is exercised as well.
    df["src_ip"] = [f"10.0.0.{i%40}" for i in range(n)]
    df["dst_ip"] = [f"10.0.1.{(i*3)%40}" for i in range(n)]
    df["ts"] = pd.date_range("2026-01-01", periods=n, freq="s")
    df["type"] = np.where(y == 1, "ddos", "normal")
    df["label"] = y
    # Deliberate missing values exercise the same cleanup path used for real data.
    df.loc[rng.choice(n, size=max(1, n//50), replace=False), "service"] = np.nan
    return df


# Helper. A browser link (github.com/.../blob/...) returns an HTML page, not the CSV.
# Rewrite it to raw.githubusercontent.com, which serves the raw file.
def _github_raw_url(url):
    """Convert a github.com .../blob/... (or .../raw/...) page link to its raw-content URL."""
    url = url.strip()
    # Capture <user>, <repo> and '<branch>/<path>' from the URL.
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/(?:blob|raw)/(.+)", url)
    if m:
        user, repo, rest = m.groups()
        return f"https://raw.githubusercontent.com/{user}/{repo}/{rest}"
    return url


# Download the CSV into memory and parse it with pandas. Any network/HTTP error is raised
# here and caught by the caller, which then falls back to local files.
def _load_from_github(url):
    raw_url = _github_raw_url(url)
    headers = {}
    # Optional token, only needed for private repositories.
    token = os.environ.get(GITHUB_TOKEN_ENV_VAR)
    if token:
        headers["Authorization"] = f"token {token}"
    req = urllib.request.Request(raw_url, headers=headers)
    # Read the whole response into memory (fine for files GitHub accepts in a normal repo,
    # i.e. under 100 MB).
    with urllib.request.urlopen(req, timeout=300) as resp:
        payload = resp.read()
    # Git LFS stores a tiny text 'pointer' in the repo instead of the real file. Detect that
    # case and explain the fix, instead of failing later with a confusing parse error.
    if payload.startswith(b"version https://git-lfs"):
        raise ValueError(
            "The URL returned a Git LFS pointer, not the CSV itself. For LFS files use "
            "https://media.githubusercontent.com/media/<user>/<repo>/<branch>/<path> instead."
        )
    if SAVE_LOCAL_COPY:
        # Download option: keep the file on disk (first DATA_CANDIDATES path) for offline reuse.
        local_copy = Path(DATA_CANDIDATES[0])
        local_copy.write_bytes(payload)
        print(f"Saved a local copy: {local_copy.resolve()}")
    return pd.read_csv(io.BytesIO(payload), low_memory=False), raw_url


# Local fallback. Checks the env-var path first, then each DATA_CANDIDATES path. Raises
# FileNotFoundError with download instructions if none exist.
def _resolve_real_csv():
    env_path = os.environ.get(DATA_ENV_VAR)
    candidates = [env_path] if env_path else []
    candidates.extend(DATA_CANDIDATES)
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate)
    raise FileNotFoundError(
        "The ToN_IoT network-flow file was not found. Download train_test_network.csv "
        "from the official UNSW ToN_IoT page: https://research.unsw.edu.au/projects/toniot-datasets. "
        "Then place it at ./train_test_network.csv or set the TON_IOT_CSV environment "
        "variable to its absolute path. The notebook will not substitute a different dataset automatically."
    )

# ---- Main load logic: self_test -> synthetic data; real -> GitHub first, then local files
# ----
if EXECUTION_MODE == "self_test":
    df = _make_self_test_dataset(RANDOM_STATE)
    DATA_SOURCE = "IN-MEMORY SELF-TEST DATA (NOT THE ToN_IoT DATASET)"
else:
    df = None
    # 1) GitHub (if GITHUB_CSV_URL is set)
    if GITHUB_CSV_URL.strip():
        try:
            df, _raw_url = _load_from_github(GITHUB_CSV_URL)
            DATA_SOURCE = f"GitHub: {_raw_url}"
        except Exception as exc:
            print(f"WARNING: could not load from GitHub ({type(exc).__name__}: {exc}). "
                  "Falling back to the local-file lookup.")
    # 2) Existing system: TON_IOT_CSV env var, then DATA_CANDIDATES paths
    if df is None:
        csv_path = _resolve_real_csv()
        df = pd.read_csv(csv_path, low_memory=False)
        DATA_SOURCE = str(csv_path.resolve())

# Record the raw size. Later cells assert that no row is lost or duplicated, which is a
# data-integrity guard.
RAW_ROW_COUNT = len(df)
RAW_COLUMN_COUNT = df.shape[1]
RAW_COLUMNS = tuple(df.columns)

# Fail-fast validation of whatever file was loaded.
if RAW_ROW_COUNT == 0:
    raise ValueError("The selected dataset contains zero rows.")
if "label" not in df.columns:
    raise ValueError("Expected binary 'label' column was not found in the selected file.")

# Always print where the data came from, so each run documents its own provenance.
print(f"Data source: {DATA_SOURCE}")
print(f"Rows: {RAW_ROW_COUNT:,}")
print(f"Columns: {RAW_COLUMN_COUNT}")
print("Label values:")
print(df["label"].value_counts(dropna=False))
df.head()


### What this cell does: preprocessing (schema cleanup only)
- Drops identifier columns (`src_ip`, `dst_ip`, `ts`) and the attack-family column (`type`), which would leak the answer or let the model memorise hosts.
- Converts the label from 0/1 to `Normal`/`Attack`.
- Fills missing text values with the category `"missing"` and missing numbers with `0`.

Nothing statistical is learned here (no means, no one-hot vocabulary), so test data cannot influence training. Encoding is deliberately postponed until after the train/test split (Section 3). The assertions confirm that no rows were lost or duplicated.

In [ ]:
# ==============================================================================
# PREPROCESSING (schema cleanup only). Drops identifier columns, converts the label to
# Normal/Attack, and fills missing values with fixed constants. Nothing is learned from the
# data here, so nothing can leak from test into train. Encoding and scaling happen later,
# after the split.
# ==============================================================================
# --- Preprocess TON_IoT into the schema the rest of the notebook expects:
#     feature columns + a 'label' column of 'Normal'/'Attack' ---

# Drop high-cardinality / leaky identifier columns and the fine-grained
# attack-type column (keep only the binary label, matching the paper's
# Normal-vs-Attack framing).
# WHY: IPs and timestamps would let a model memorise specific hosts/times (leakage), and
# 'type' is the attack family, which reveals the label.
drop_cols = [c for c in ["src_ip", "dst_ip", "ts", "type"] if c in df.columns]
data = df.drop(columns=drop_cols)

# Binary label: TON_IoT already ships 'label' as 0/1 (0=Normal, 1=Attack)
# 0/1 -> 'Normal'/'Attack'. Any other value becomes NaN and is caught by the assertion
# below.
data["label"] = data["label"].map({0: "Normal", 1: "Attack"})

# Standardize missing values in the categorical (object-dtype) columns (proto, service,
# conn_state, ssl/http/dns fields, etc.) to an explicit "missing" token. This is schema
# cleanup only: no encoding vocabulary or fitted statistic is created here.
#
# One-hot encoding is deferred to Section 3, where it is fit on the TRAINING split only and
# then applied to the test split with that vocabulary frozen (test-only categories are
# dropped; categories absent from a row are zero-filled). This keeps the vocabulary from
# being derived from rows that later become the test set.
#
# text_cols = every non-numeric column except the label.
text_cols = [c for c in data.columns
            if not pd.api.types.is_numeric_dtype(data[c]) and c != "label"]
# Missing categorical value -> explicit 'missing' category, so the row is kept.
data[text_cols] = data[text_cols].fillna("missing")

# Fill any remaining numeric NaNs (TON_IoT has some missing DNS/HTTP/SSL
# fields) with a fixed constant (0, meaning "field not applicable/absent").
# This is a domain-motivated constant, not a statistic estimated from the
# data (e.g. a train-set mean/median), so applying it before the split does
# not leak any train/test distributional information.
# num_cols = all remaining columns except the label.
num_cols = [c for c in data.columns if c not in text_cols and c != "label"]
# Missing numeric value -> 0. A constant (not a mean/median), so no statistic is estimated
# from the data.
data[num_cols] = data[num_cols].fillna(0)

# --- Use the complete loaded dataset ---
# Every available row is kept after schema cleanup. Section 3 splits the COMPLETE dataset into
# train/test before fitting the one-hot vocabulary on the training split only.

print(f"Using all available rows: {len(data):,}")
# ---- Integrity checks: preprocessing must not add/remove rows or leave unknown/missing
# labels ----
assert len(data) == RAW_ROW_COUNT, (
    f"Row-count changed during schema preprocessing: raw={RAW_ROW_COUNT}, "
    f"preprocessed={len(data)}"
)
assert len(data.index.unique()) == RAW_ROW_COUNT, "Duplicate row index detected before splitting."
assert data["label"].notna().all(), "Missing labels detected after preprocessing."
assert set(data["label"].unique()).issubset({"Normal", "Attack"}), \
    f"Unexpected labels: {sorted(data['label'].dropna().unique().tolist())}"

# Keep the complete pre-encoding dataframe in memory. One-hot encoding happens
# later, split-safe, in Section 3. Avoid writing and re-reading another copy of
# the dataset so the experiment has one authoritative in-memory source.
# From here on `df` is the cleaned, still-unencoded dataset that Section 3 will split.
df = data
print(f"Full dataset preserved in memory: {len(df):,} rows x {df.shape[1]-1} feature columns")
df.head()


## 2. Models: 4 baselines + Trust-Drift Fusion (TDF)

### What this cell does: defines all models
- **Static Threshold**: alarm if any feature is more than `k` std-devs from the Normal average.
- **Naive Bayes** and **Logistic Regression**: standard supervised baselines.
- **Isolation Forest**: unsupervised anomaly detector (ignores labels).
- **Trust-Drift Fusion (proposed)**: fuses AS + RDS + RF with weights found by grid search on a validation split, then chooses a threshold (best F1, or lowest cost).

Also contains helper functions for fast threshold search and for turning a cost model into severity bands. Every model exposes `fit()` and `score()` (0 to 1, higher = more attack-like) so they can be compared with one routine.

In [ ]:
# ==============================================================================
# MODEL ZOO. Four baselines plus the proposed Trust-Drift Fusion (TDF) model. All share one
# interface: fit(X, y) and score(X) -> value in [0, 1] (higher = more attack-like), so a
# single evaluation routine can compare them fairly.
# ==============================================================================
"""
Defines the 4 baseline models and the proposed Trust-Drift Fusion (TDF)
model used in Section IV of the paper. Each model exposes:
    fit(X_train, y_train)
    score(X_test) -> continuous risk score in [0, 1] (higher = more attack-like)
so that a common threshold-sweep / metric routine can evaluate all of them.
"""


# BASELINE 1: a classic rule. Raise an alarm when ANY single feature is more than k standard
# deviations away from the Normal average.
class StaticThresholdModel:
    """Flags a window as Attack if any single feature exceeds
    mean + k*std learned from the training (mostly-normal) partition."""

    def __init__(self, k=3.0):
        self.k = k

    def fit(self, X, y):
        # Learn what 'normal' looks like from Normal rows only.
        normal = X[y == 0]
        self.mu = normal.mean(axis=0)
        # +1e-9 prevents division by zero for constant features.
        self.sigma = normal.std(axis=0) + 1e-9
        return self

    def score(self, X):
        # z-score per feature = how many standard deviations from normal.
        z = np.abs((X - self.mu) / self.sigma)
        # Take the worst feature per row (this is what makes it a 'single feature exceeds'
        # rule).
        max_z = z.max(axis=1)
        # Scale to [0, 1]: a worst-case z-score of 2*k maps to 1.0.
        return np.clip(max_z / (self.k * 2), 0, 1)


# BASELINE 2: Gaussian Naive Bayes on standardised features (assumes features are
# independent given the class).
class NaiveBayesModel:
    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        self.clf = GaussianNB().fit(self.scaler.transform(X), y)
        return self

    def score(self, X):
        return self.clf.predict_proba(self.scaler.transform(X))[:, 1]


# BASELINE 3: unsupervised anomaly detector. It ignores the labels y while fitting.
class IsolationForestModel:
    """Standalone unsupervised Isolation Forest (no drift/policy fusion)."""

    def __init__(self, n_estimators=100, contamination=0.19, random_state=RANDOM_STATE):
        self.n_estimators = n_estimators
        self.contamination = contamination
        self.random_state = random_state

    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        self.model = IsolationForest(
            n_estimators=self.n_estimators,
            contamination=self.contamination,
            random_state=self.random_state,
        ).fit(self.scaler.transform(X))
        # score_samples is HIGHER for normal points, so negate it: higher = more anomalous.
        raw = -self.model.score_samples(self.scaler.transform(X))
        # Remember the training min/max so test scores can be rescaled consistently.
        self.rmin, self.rmax = raw.min(), raw.max()
        return self

    def score(self, X):
        raw = -self.model.score_samples(self.scaler.transform(X))
        # Min-max scale with the TRAINING range; clip handles test values outside that
        # range.
        return np.clip((raw - self.rmin) / (self.rmax - self.rmin + 1e-9), 0, 1)


# BASELINE 4: linear classifier on standardised features.
class LogisticRegressionModel:
    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        self.clf = LogisticRegression(max_iter=1000).fit(
            self.scaler.transform(X), y
        )
        return self

    def score(self, X):
        return self.clf.predict_proba(self.scaler.transform(X))[:, 1]


# THE PROPOSED MODEL. Fuses three signals: AS (Isolation Forest anomaly), RDS (drift away
# from the Normal baseline) and RF (Random Forest attack probability): HT = w_as*AS +
# w_rds*RDS + w_rf*RF. The weights and threshold are learned on a validation split.
class TrustDriftFusionModel:
    """Proposed model: fuses an Isolation Forest anomaly score (AS), an
    training-reference resource-drift score (RDS, Z-score based), and a Random-Forest
    attack probability (RF) into a single Hybrid Threat score via weighted
    fusion (Eq. 4 in the paper). Fusion weights are auto-tuned by grid
    search on a held-out validation split of the training data, rather
    than fixed by hand."""

    # Defaults come from the configuration cell. They are bound when this cell runs, so re-run
    # this cell after changing any of those settings.
    def __init__(self, n_estimators=N_ESTIMATORS, contamination=0.19,
                 random_state=RANDOM_STATE, val_size=VALIDATION_SIZE,
                 weight_step=WEIGHT_STEP, min_weight=0.0, rf_max_depth=RF_MAX_DEPTH,
                 threshold_mode="f1", cost_fn=5.0, cost_fp=1.0):
        self.n_estimators = n_estimators
        self.contamination = contamination
        self.random_state = random_state
        self.val_size = val_size
        self.weight_step = weight_step
        self.min_weight = min_weight
        self.rf_max_depth = rf_max_depth
        # threshold_mode: "f1" (default, maximizes F1) or "cost" (minimizes
        # cost_fn*FN + cost_fp*FP -- e.g. a missed attack costing 5x more
        # than a false quarantine action, reflecting real operational risk
        # rather than treating both error types as equally bad).
        self.threshold_mode = threshold_mode
        self.cost_fn = cost_fn
        self.cost_fp = cost_fp

    def fit(self, X, y):
        # ---- Input validation: fail early with clear error messages ----
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        if X.ndim != 2 or len(X) != len(y):
            raise ValueError("X must be 2D and have the same number of rows as y.")
        if len(X) < 10:
            raise ValueError("Not enough rows for a stable train/validation split.")
        if np.unique(y).size < 2:
            raise ValueError("TDF requires both Normal and Attack labels in the training data.")
        if not np.isfinite(X).all():
            raise ValueError("Non-finite feature values detected before TDF fitting.")

        # Hold out a validation split from the training data only (test set
        # is never touched here) to tune the fusion weights.
        # X_sub/y_sub (75%) fit the three detectors. X_val/y_val (25%) tune the fusion
        # weights and threshold. The test set is never seen here.
        X_sub, X_val, y_sub, y_val = train_test_split(
            X, y, test_size=self.val_size, random_state=self.random_state,
            stratify=y if y.sum() > 1 else None,
        )
        assert len(X_sub) + len(X_val) == len(X)
        assert np.unique(y_sub).size == 2 and np.unique(y_val).size == 2, \
            "Training and validation partitions must both contain both classes."

        # Standardise features (mean 0, std 1) using the sub-train part only.
        self.scaler = StandardScaler().fit(X_sub)
        Xs_sub = self.scaler.transform(X_sub)

        # Anomaly signal (unsupervised)
        self.iso = IsolationForest(
            n_estimators=self.n_estimators,
            contamination=self.contamination,
            random_state=self.random_state,
        ).fit(Xs_sub)
        # Remember the training anomaly range so AS can later be scaled to [0, 1].
        # (contamination = assumed share of anomalies, a fixed hyperparameter.)
        raw = -self.iso.score_samples(Xs_sub)
        self.as_min, self.as_max = raw.min(), raw.max()

        # Resource-drift reference (Normal windows only; fixed after training)
        normal = X_sub[y_sub == 0]
        self.mu = normal.mean(axis=0)
        self.sigma = normal.std(axis=0) + 1e-9

        # Supervised signal: real Random Forest (matches Eq. 3 in the paper)
        # class_weight='balanced' re-weights the classes so the smaller class is not
        # ignored.
        self.rf = RandomForestClassifier(
            n_estimators=self.n_estimators, random_state=self.random_state,
            max_depth=self.rf_max_depth, class_weight="balanced",
        ).fit(Xs_sub, y_sub)

        # --- Grid-search fusion weights on the validation split ---
        AS_v, RDS_v, RF_v = self._components(X_val)
        # Start with a fallback (weights 0.35/0.25/0.40, threshold 0.5); any real grid point
        # overrides it.
        best_score, best_w, best_t = -1e18, (0.35, 0.25, 0.40), 0.5
        step = self.weight_step
        # Candidate weights are tried on a grid of step `weight_step` (0.05).
        grid = np.arange(0, 1 + 1e-9, step)
        mw = self.min_weight
        for w_as in grid:
            for w_rds in grid:
                # The third weight is implied so the three always sum to 1.
                w_rf = 1 - w_as - w_rds
                if w_rf < 0 or w_rf > 1:
                    continue
                if mw > 0 and (w_as < mw or w_rds < mw or w_rf < mw):
                    continue  # enforce every signal keeps a meaningful floor
                # Fused Hybrid Threat score for this weight combination on the validation
                # data.
                HT = w_as * AS_v + w_rds * RDS_v + w_rf * RF_v
                # Best threshold for this combination and its objective value (F1, or
                # negative cost in cost mode).
                t, score = self._select_threshold(y_val, HT)
                # Keep the best (weights, threshold) over the whole grid.
                if score > best_score:
                    best_score, best_w, best_t = score, (w_as, w_rds, w_rf), t
        # Freeze the learned configuration. No test data was used to get here.
        self.w_as, self.w_rds, self.w_rf = best_w
        self.threshold = best_t

        # Expose the validation split (and its component scores) so other
        # tools (ablation study, cost-threshold analysis, etc.) can reuse
        # the exact same leakage-free split rather than re-deriving it.
        self.X_val_, self.y_val_ = X_val, y_val
        self.AS_val_, self.RDS_val_, self.RF_val_ = AS_v, RDS_v, RF_v
        return self

    # Choose the decision threshold on validation scores: minimise cost in 'cost' mode,
    # otherwise maximise F1.
    def _select_threshold(self, y_true, scores):
        """Returns (threshold, objective_value) using whichever selection
        mode this model was configured with."""
        if self.threshold_mode == "cost":
            return best_cost_threshold(y_true, scores, self.cost_fn, self.cost_fp)
        t = best_f1_threshold_local(y_true, scores)
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        return t, f1

    # Computes the three raw signals AS, RDS, RF for any X, each scaled to [0, 1]. Reused by
    # the ablation, context-aware, SHAP and drift code.
    def _components(self, X):
        Xs = self.scaler.transform(X)
        raw = -self.iso.score_samples(Xs)
        AS = np.clip((raw - self.as_min) / (self.as_max - self.as_min + 1e-9), 0, 1)

        # RDS: average distance from the Normal baseline (mean |z-score|), divided by 4 and
        # clipped, so an average drift of 4 std-devs saturates at 1.
        z = np.abs((X - self.mu) / self.sigma)
        RDS = np.clip(z.mean(axis=1) / 4.0, 0, 1)

        RF = self.rf.predict_proba(Xs)[:, 1]
        return AS, RDS, RF

    def score(self, X):
        AS, RDS, RF = self._components(X)
        # Weighted fusion -> final risk score in [0, 1].
        HT = self.w_as * AS + self.w_rds * RDS + self.w_rf * RF
        return np.clip(HT, 0, 1)


# PERFORMANCE HELPER. Counts TP / FP / FN for all 99 thresholds (0.01 ... 0.99) in a single
# pass using sorting + histograms instead of 99 separate predictions. Same answer as a naive
# loop, just much faster on 200k+ rows.
def _threshold_grid_stats(y_true, scores, thresholds=None):
    """Compute TP/FP/FN for the notebook's fixed threshold grid in O(n + k) time.

    This is exactly equivalent to evaluating every threshold in ascending
    order, but avoids creating a full (rows × thresholds) prediction matrix.
    """
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=float)
    thresholds = (np.linspace(0.01, 0.99, 99)
                  if thresholds is None else np.asarray(thresholds, dtype=float))
    # k[i] = number of grid thresholds <= scores[i].
    # For each score: how many grid thresholds are <= it. That decides for which thresholds
    # it counts as 'predicted attack'.
    k = np.searchsorted(thresholds, scores, side="right")
    # Histograms of all rows and of true attacks per bucket. Reverse cumulative sums turn
    # them into counts for each threshold.
    hist_all = np.bincount(k, minlength=len(thresholds) + 1)
    hist_pos = np.bincount(k, weights=y_true, minlength=len(thresholds) + 1)
    # For threshold j, predicted-positive rows are buckets j+1 ... end.
    pred_pos = np.cumsum(hist_all[::-1])[::-1][1:]
    tp = np.cumsum(hist_pos[::-1])[::-1][1:]
    total_pos = float(y_true.sum())
    # FP = predicted positives that are not true positives; FN = true positives that were
    # missed.
    fp = pred_pos - tp
    fn = total_pos - tp
    return thresholds, tp, fp, fn


# Pick the threshold (0.01 ... 0.99) with the best F1 score.
def best_f1_threshold_local(y_true, scores):
    """Return the same 0.01..0.99 fixed-grid F1-optimal threshold as before,
    but compute all 99 candidates without 99 full prediction passes."""
    thresholds, tp, fp, fn = _threshold_grid_stats(y_true, scores)
    # F1 = 2TP / (2TP + FP + FN); the guard avoids 0/0.
    denom = 2.0 * tp + fp + fn
    f1 = np.divide(2.0 * tp, denom, out=np.zeros_like(tp, dtype=float), where=denom > 0)
    # np.argmax returns the first maximum, matching the original ascending loop.
    return float(thresholds[int(np.argmax(f1))])


# Pick the threshold with the lowest total cost, where a missed attack costs `cost_fn` and a
# false alarm costs `cost_fp`.
def best_cost_threshold(y_true, scores, cost_fn=5.0, cost_fp=1.0):
    """Return the same fixed-grid threshold minimizing cost_fn*FN + cost_fp*FP,
    using a vectorized confusion-count calculation. A negative minimum cost is
    returned so callers can continue to maximize the objective exactly as before."""
    thresholds, _tp, fp, fn = _threshold_grid_stats(y_true, scores)
    # Total cost at every threshold.
    costs = cost_fn * fn + cost_fp * fp
    # First minimum preserves the original ascending-threshold tie behavior.
    return float(thresholds[int(np.argmin(costs))]), -float(np.min(costs))


# Turns the cost model into Trust-Score cutoffs (one per cost ratio) instead of hard-coded
# constants.
def severity_bands_from_cost(model, cost_schedule=(1.0, 3.0, 8.0)):
    """Derives the 3 severity-band cutoffs (Trusted/Monitored, Monitored/
    Elevated, Elevated/Critical) from an escalating cost schedule instead
    of the fixed 80/60/35 constants, using the model's own stored
    validation split. cost_schedule gives the false-negative:false-positive
    cost ratio required to justify each successive escalation tier (e.g.
    1.0 = "Monitor vs Log&Watch" boundary treats both errors equally;
    8.0 = "Isolate vs Quarantine&Escalate" boundary requires a missed
    attack to be judged 8x worse than an unnecessary quarantine before the
    system will escalate that far). Returns 3 Trust-Score cutoffs (0-100),
    highest first, matching the T>=tau1 / tau2<=T<tau1 / tau3<=T<tau2 /
    T<tau3 structure in Eq. 6 of the paper.
    """
    AS_v, RDS_v, RF_v = model.AS_val_, model.RDS_val_, model.RF_val_
    HT_v = model.w_as * AS_v + model.w_rds * RDS_v + model.w_rf * RF_v
    cutoffs = []
    for ratio in cost_schedule:
        # Best HT threshold for this FN:FP cost ratio.
        t_ht, _ = best_cost_threshold(model.y_val_, HT_v, cost_fn=ratio, cost_fp=1.0)
        trust_cutoff = 100 - 100 * t_ht  # HT -> Trust Score, same mapping as Eq. 5
        cutoffs.append(trust_cutoff)
    return sorted(cutoffs, reverse=True)


# Name -> class. The evaluation loop iterates this dict, so adding a model = adding one
# line.
MODEL_REGISTRY = {
    "Static Threshold": StaticThresholdModel,
    "Naive Bayes": NaiveBayesModel,
    "Isolation Forest": IsolationForestModel,
    "Logistic Regression": LogisticRegressionModel,
    "TDF (Proposed)": TrustDriftFusionModel,
}


**Note on the standalone Isolation Forest baseline.** `IsolationForestModel` and the anomaly-score (AS) branch inside `TrustDriftFusionModel` both fit `IsolationForest` on the sub-train partition, which contains *both* classes, not on Normal-only rows. Because Isolation Forest is unsupervised, it treats whichever population is easiest to isolate as the "anomaly". If Attack rows are the majority of the training data (check the class balance printed in Section 1), the minority Normal rows are the ones that look anomalous, which is the opposite of the Attack-vs-Normal target. In that case the standalone Isolation Forest can score *below* random guessing (ROC-AUC under 0.50; see Table 1). This is not a bug in the fusion pipeline: the resource-drift channel (RDS) is fit on Normal-only rows as intended, and the weight search simply gives AS a small weight when its signal is weak or inverted (the learned weights are printed in Section 4). Worth stating explicitly in the paper so a reader does not mistake a sub-random AUC for a measurement error.


## 3. Baseline comparison (Table 1 + figures)

**Protocol note (test-set leakage control):** every model's decision threshold is chosen on a validation split carved out of the training data, never on the test set. Each baseline gets its own train/validation split (same `VALIDATION_SIZE` / `RANDOM_STATE` convention as `TrustDriftFusionModel`), is fit on the sub-train partition only, and has its threshold frozen on the validation partition. TDF does this internally, and its own frozen `model.threshold` is used as is. `y_test` is used only in the final metrics block.


**Protocol note (preprocessing leakage control):** Section 1 only cleans missing values and drops identifier columns; categorical columns stay as strings. The cell below splits the raw rows into train/test first, then fits the one-hot vocabulary on the training rows only (`pd.get_dummies` on `train_df`) and applies that frozen vocabulary to the test rows via `reindex` (test-only categories are dropped; categories absent from a test row are zero-filled).


### What this cell does: the main benchmark
1. Split the raw rows 70/30 into train/test (stratified).
2. One-hot encode text columns using the **training** rows only; apply that frozen vocabulary to the test rows.
3. Fit all five models. Each model's threshold is chosen on a validation part of the training data, never on the test set.
4. Compute eight metrics on the test set and save them as `table1_results.csv`.

The last block prints ROC-AUC and Average Precision to 5 decimals so the headline numbers are traceable.

In [ ]:
# ==============================================================================
# CORE EXPERIMENT (Section 3). Leakage-safe pipeline: split first -> encode using training
# rows only -> fit all 5 models -> evaluate once on the held-out test set -> Table II.
# ==============================================================================
# Columns that must never be used as model input (identifiers and label columns).
NON_FEATURE_COLS = {"device_id", "profile", "window", "attack_type", "label", "id", "attack_cat"}

# --- Leakage-safe categorical encoding: split the RAW (pre-one-hot) rows into
# train/test FIRST, then fit the one-hot vocabulary on the training rows only
# and apply that frozen vocabulary to the test rows (see the correction note
# above -- Section 1 no longer encodes before the split). This mirrors the
# same train-only-fit / apply-to-test discipline already used below for
# scaling and for fusion-weight/threshold tuning.
# Text columns that need one-hot encoding.
cat_cols = [c for c in df.columns
            if c not in NON_FEATURE_COLS and not pd.api.types.is_numeric_dtype(df[c])]

# Stratified train/test split (TEST_SIZE, default 30%). `stratify` keeps the Normal/Attack ratio
# the same in both parts. This is THE split that defines the held-out test set.
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"]
)

# Learn the dummy-column vocabulary from the TRAINING rows.
def _one_hot_fit(frame):
    return pd.get_dummies(frame, columns=cat_cols, dummy_na=False) if cat_cols else frame.copy()

# Encode other rows using that frozen vocabulary.
def _one_hot_apply(frame, train_columns):
    enc = pd.get_dummies(frame, columns=cat_cols, dummy_na=False) if cat_cols else frame.copy()
    # Freeze the TRAINING vocabulary: any dummy column seen in training but
    # absent here is zero-filled; any category value that only appears in
    # this split (e.g. a test-only category) is dropped, since no
    # corresponding column existed when the models were fit.
    return enc.reindex(columns=train_columns, fill_value=0)

# Encode train (defines the columns), then test (forced to have exactly the same columns).
train_enc = _one_hot_fit(train_df)
test_enc = _one_hot_apply(test_df, train_enc.columns)

# Final model inputs = numeric columns of the encoded training frame.
feature_cols = [c for c in train_enc.columns
                if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(train_enc[c])]
print(f"Using {len(feature_cols)} feature columns (one-hot vocabulary fit on the training split only)")
# ---- Integrity checks: no row lost, duplicated, or shared between train and test ----
assert len(train_df) + len(test_df) == RAW_ROW_COUNT, "Train/test rows do not conserve the complete dataset."
assert set(train_df.index).isdisjoint(set(test_df.index)), "Train/test row overlap detected."
assert len(set(train_df.index) | set(test_df.index)) == RAW_ROW_COUNT, "Some source rows are missing from train/test."

# Convert to NumPy arrays. Labels become 0/1 with Attack = 1.
X_train = train_enc[feature_cols].values.astype(float)
X_test = test_enc[feature_cols].values.astype(float)
y_train = (train_enc["label"] == "Attack").astype(int).values
y_test = (test_enc["label"] == "Attack").astype(int).values

# Reproducibility record: the train/validation split used for baseline
# threshold selection and by TrustDriftFusionModel.fit(). This creates indices
# only for reporting; the models below use the same deterministic split directly.
protocol_train_idx, protocol_val_idx = train_test_split(
    np.arange(len(X_train)), test_size=VALIDATION_SIZE, random_state=RANDOM_STATE,
    stratify=y_train if y_train.sum() > 1 else None,
)

def _dist(y):
    values, counts = np.unique(y, return_counts=True)
    return {int(v): int(c) for v, c in zip(values, counts)}

# Report the three-way partition actually used: sub-train / validation / test.
print("\n=== Final data split (reproducibility) ===")
print(f"Train:      {len(protocol_train_idx):,} rows | Normal={np.sum(y_train[protocol_train_idx] == 0):,} | Attack={np.sum(y_train[protocol_train_idx] == 1):,}")
print(f"Validation: {len(protocol_val_idx):,} rows | Normal={np.sum(y_train[protocol_val_idx] == 0):,} | Attack={np.sum(y_train[protocol_val_idx] == 1):,}")
print(f"Test:       {len(y_test):,} rows | Normal={np.sum(y_test == 0):,} | Attack={np.sum(y_test == 1):,}")

# Lightweight runtime invariants catch accidental corruption immediately.
assert len(X_train) + len(X_test) == RAW_ROW_COUNT, "Train/test row count mismatch against raw dataset."
assert X_train.shape[1] == X_test.shape[1] == len(feature_cols), "Feature-column mismatch."
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "Non-finite values remain after preprocessing."
assert set(np.unique(y_train)).issubset({0, 1}) and set(np.unique(y_test)).issubset({0, 1}), "Labels are not binary."


# Fit one model, freeze its threshold WITHOUT the test set, then compute metrics on the test
# set. y_test is only touched in the last block.
def evaluate_model(name, model_cls, X_train, y_train, X_test, y_test):
    """Fit + threshold-select a model with y_test used ONLY for the final
    metrics returned at the end -- never for choosing a threshold, a weight,
    or a hyperparameter.

    - TrustDriftFusionModel already carves its own validation split out of
      (X_train, y_train) inside .fit() and freezes self.threshold (and its
      fusion weights) using that validation split. For TDF we simply reuse
      that already-leakage-free, already-frozen threshold as-is.
    - None of the four baseline classes (StaticThresholdModel, NaiveBayesModel,
      IsolationForestModel, LogisticRegressionModel) do this internally, so we
      carve out the same kind of validation split here, fit the baseline on
      the sub-train partition only, and select+freeze its decision threshold
      by maximizing F1 on the held-out validation partition. y_test is never
      touched until the final metrics block below.
    """
    # TDF creates its own validation split inside fit() and freezes its own threshold, so we
    # reuse it as is.
    if model_cls is TrustDriftFusionModel:
        model = model_cls().fit(X_train, y_train)
        t = model.threshold
    else:
        # Baselines: same recipe as TDF. Fit on the sub-train part, choose the F1-optimal threshold
        # on the held-out validation part.
        X_sub, X_val, y_sub, y_val = train_test_split(
            X_train, y_train, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE,
            stratify=y_train if y_train.sum() > 1 else None,
        )
        model = model_cls().fit(X_sub, y_sub)
        val_scores = model.score(X_val)
        t = best_f1_threshold_local(y_val, val_scores)

    # --- Final evaluation only, from here down: y_test is used exclusively
    # to report metrics against the frozen threshold `t`, never to choose it. ---
    scores = model.score(X_test)
    pred = (scores >= t).astype(int)
    # Eight metrics in percent. Threshold-based: Accuracy, Precision, Recall, F1, MCC,
    # Balanced Accuracy. Threshold-free (use raw scores): ROC-AUC, PR-AUC.
    metrics = {
        "Accuracy": accuracy_score(y_test, pred) * 100,
        "Precision": precision_score(y_test, pred, zero_division=0) * 100,
        "Recall": recall_score(y_test, pred, zero_division=0) * 100,
        "F1-Score": f1_score(y_test, pred, zero_division=0) * 100,
        "ROC-AUC": roc_auc_score(y_test, scores) * 100,
        "PR-AUC": average_precision_score(y_test, scores) * 100,
        "MCC": matthews_corrcoef(y_test, pred) * 100,
        "Balanced Accuracy": balanced_accuracy_score(y_test, pred) * 100,
    }
    return metrics, scores, pred, model

# Run all five models and keep metrics, scores, predictions and fitted objects. Later
# sections (ablation, SHAP, adversarial) reuse `fitted_models`.
results, scores_by_model, preds_by_model, fitted_models = {}, {}, {}, {}
for name, cls in MODEL_REGISTRY.items():
    metrics, scores, pred, model = evaluate_model(name, cls, X_train, y_train, X_test, y_test)
    results[name], scores_by_model[name], preds_by_model[name], fitted_models[name] = metrics, scores, pred, model
    print(f"{name:22s} " + "  ".join(f"{k}={v:6.2f}" for k, v in metrics.items()))

# Table 1 (rows = models, columns = metrics), saved to CSV. The final certification cell
# re-reads this file.
table = pd.DataFrame(results).T
table.to_csv("table1_results.csv")
print("\n=== Table ===")
table.round(2)


# --- High-precision headline metrics (5 decimals) -----------------------------
# The abstract quotes TDF ROC-AUC and average precision to 5 decimal places
# (0.99991 / 0.99996). The table above only prints 2-decimal percentages, so
# those exact abstract figures were not directly traceable to a printed
# number. This makes them traceable.
tdf_scores_final = scores_by_model["TDF (Proposed)"]
print("\n=== High-precision headline metrics (TDF, held-out test set) ===")
print(f"ROC-AUC:           {roc_auc_score(y_test, tdf_scores_final):.5f}")
print(f"Average Precision: {average_precision_score(y_test, tdf_scores_final):.5f}")


### Figures: confusion matrix, PR curve, ROC curve
Three plots built from the test-set scores produced above. They are saved as image files that the final certification cell checks for.

In [ ]:
# ==============================================================================
# CONFUSION MATRIX for the proposed TDF model on the test set. Layout: rows = true class,
# columns = predicted class, i.e. [[TN, FP], [FN, TP]].
# ==============================================================================
# Confusion matrix (proposed model)
cm = confusion_matrix(y_test, preds_by_model["TDF (Proposed)"])
fig, ax = plt.subplots(figsize=(4.5, 4.0))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Attack"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Attack"])
ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")
# Print the count inside each cell (white text on dark cells for readability).
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout();
# Saved as vector PDF (for the paper) and 600-dpi PNG (checked by the certification cell).
plt.savefig("confusion_matrix.pdf", bbox_inches="tight")
plt.savefig("confusion_matrix.png", dpi=600, bbox_inches="tight");
plt.show()


In [ ]:
# ==============================================================================
# PRECISION-RECALL CURVES for all models on the test set. More informative than ROC when the
# classes are imbalanced.
# ==============================================================================

# Precision-Recall curves (final held-out test evaluation only)
fig, ax = plt.subplots(figsize=(5.0, 4.2))
for name in MODEL_REGISTRY:
    # One curve per model across all thresholds. AP (average precision) = area under this
    # curve.
    precision, recall, _ = precision_recall_curve(y_test, scores_by_model[name])
    pr_auc = average_precision_score(y_test, scores_by_model[name])
    ax.plot(recall, precision, label=f"{name} (AP={pr_auc:.3f})", linewidth=1.5)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves")
ax.legend(fontsize=6, loc="lower left")
plt.tight_layout()
plt.savefig("pr_curve.png", dpi=600)
plt.show()


In [ ]:
# ==============================================================================
# ROC CURVES for all models on the test set (true-positive rate vs false-positive rate
# across thresholds).
# ==============================================================================
# ROC curves
fig, ax = plt.subplots(figsize=(5.0, 4.2))
for name in MODEL_REGISTRY:
    fpr, tpr, _ = roc_curve(y_test, scores_by_model[name])
    # Reuse the AUC from Table 1 so the figure and the table can never disagree.
    auc = results[name]["ROC-AUC"] / 100
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.5)
# Diagonal = random guessing (AUC 0.5).
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Guess")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=6, loc="lower right")
plt.tight_layout(); plt.savefig("roc_curve.png", dpi=600); plt.show()


## 4. Ablation study
What does each fused signal (AS, RDS, RF) contribute on its own, in every pairwise
combination, and in the full fusion - all sharing the exact same fitted detectors,
so any difference is attributable to the *combination strategy*, not different models.

### What this cell does: ablation study
Asks "does each signal earn its place?". The same fitted detectors are scored with different weights: AS only, RDS only, RF only, each equal-weight pair, and the full learned fusion. Thresholds are picked on validation data, metrics reported on test. Results go to `ablation_results.csv`.

In [ ]:
# ==============================================================================
# ABLATION STUDY (Section 4). Which of the three signals matters? Score the SAME fitted
# detectors with different weightings: each signal alone, equal pairs, and the full learned
# fusion.
# ==============================================================================

# Each entry is (w_as, w_rds, w_rf).
COMBOS_FIXED = {
    "AS only":         (1.0, 0.0, 0.0),
    "RDS only":        (0.0, 1.0, 0.0),
    "RF only":         (0.0, 0.0, 1.0),
    "AS + RDS (equal)": (0.5, 0.5, 0.0),
    "AS + RF (equal)":  (0.5, 0.0, 0.5),
    "RDS + RF (equal)": (0.0, 0.5, 0.5),
}

# The same 8 metrics as Table 1, for any score vector and threshold.
def metrics_for(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred) * 100,
        "Precision": precision_score(y_true, pred, zero_division=0) * 100,
        "Recall": recall_score(y_true, pred, zero_division=0) * 100,
        "F1-Score": f1_score(y_true, pred, zero_division=0) * 100,
        "ROC-AUC": (roc_auc_score(y_true, scores) * 100 if len(np.unique(y_true)) > 1 else float("nan")),
        "PR-AUC": average_precision_score(y_true, scores) * 100,
        "MCC": matthews_corrcoef(y_true, pred) * 100,
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred) * 100,
    }

# Reuse the SAME fitted TDF, so any difference is caused by how the signals are combined,
# not by different models.
tdf_model = fitted_models["TDF (Proposed)"]
# Signals on the validation set (used to choose thresholds) ...
AS_val, RDS_val, RF_val = tdf_model.AS_val_, tdf_model.RDS_val_, tdf_model.RF_val_
y_val = tdf_model.y_val_
# ... and on the test set (used only to report metrics).
AS_test, RDS_test, RF_test = tdf_model._components(X_test)

rows = []
# For every combination: choose the best-F1 threshold on VALIDATION, then report metrics on
# TEST.
for name, (w_as, w_rds, w_rf) in COMBOS_FIXED.items():
    HT_val = w_as * AS_val + w_rds * RDS_val + w_rf * RF_val
    t = best_f1_threshold_local(y_val, HT_val)

    HT_test = w_as * AS_test + w_rds * RDS_test + w_rf * RF_test
    m = metrics_for(y_test, HT_test, t)
    rows.append({
        "Combination": name,
        "w_as": w_as,
        "w_rds": w_rds,
        "w_rf": w_rf,
        **m,
    })

# Last row of the table: the full TDF with its learned weights and threshold.
HT_test_full = tdf_model.w_as * AS_test + tdf_model.w_rds * RDS_test + tdf_model.w_rf * RF_test
m_full = metrics_for(y_test, HT_test_full, tdf_model.threshold)
rows.append({
    "Combination": "Full TDF (learned weights)",
    "w_as": tdf_model.w_as,
    "w_rds": tdf_model.w_rds,
    "w_rf": tdf_model.w_rf,
    **m_full,
})

# Saved to CSV (also checked by the certification cell).
ablation_result = pd.DataFrame(rows)
ablation_result.to_csv("ablation_results.csv", index=False)
print(f"Full-fusion learned weights: w_as={tdf_model.w_as:.2f} w_rds={tdf_model.w_rds:.2f} w_rf={tdf_model.w_rf:.2f}\n")
ablation_result.round(2)


## 5. Context-aware (per-regime) fusion weights
Regime discovery and per-regime weight/threshold optimization use **disjoint**
development subsets. The final held-out test set is never used during context
discovery or tuning. With no natural device-profile field in the processed
TON_IoT table, regimes are discovered by KMeans on the frozen AS/RDS/RF component
space. This is a context-aware extension of TDF, not a redefinition of the base
TDF architecture.


### What this cell does: context-aware TDF (class definition)
One global weight vector may not suit every kind of traffic. This class discovers regimes (KMeans on the AS/RDS/RF space) and tunes weights and thresholds per regime. Training data is split into three disjoint parts (fit the base model / discover regimes / tune regimes) so no row is used twice for conflicting purposes.

In [ ]:
# ==============================================================================
# CONTEXT-AWARE TDF (Section 5). Instead of one global weight vector, discover traffic
# 'regimes' and tune weights + threshold per regime. Data is split three ways so regime
# discovery, regime tuning and base-model fitting never share rows.
# ==============================================================================


# Idea: different traffic regimes may rely on different signals (e.g. RF is more useful in
# one regime, RDS in another).
class ContextAwareTDF:
    """Context-aware TDF with disjoint regime-discovery and tuning sets.

    Protocol:
      1. Split the supplied training partition into:
         - model-fit portion,
         - regime-discovery portion,
         - regime-tuning portion.
      2. Fit the shared base TDF only on the model-fit portion. Its own internal
         validation split remains disjoint from both context holdouts.
      3. Discover regimes on the regime-discovery portion only.
      4. Optimize per-regime fusion weights + thresholds on the separate
         regime-tuning portion only.
      5. The final held-out test set is never passed to fit(), discovery,
         weighting, threshold selection, or hyperparameter selection.

    This avoids using the same validation observations to both discover regimes
    and optimize their decision rules.
    """

    def __init__(
        self,
        n_clusters=3,
        base_kwargs=None,
        weight_step=WEIGHT_STEP,
        min_weight=0.0,
        random_state=RANDOM_STATE,
        context_holdout=0.40,
    ):
        self.n_clusters = n_clusters
        self.base_kwargs = base_kwargs or {}
        self.weight_step = weight_step
        self.min_weight = min_weight
        self.random_state = random_state
        self.context_holdout = context_holdout

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of rows.")
        if not (0 < self.context_holdout < 1):
            raise ValueError("context_holdout must be between 0 and 1.")

        # Split EXPLICIT ROW INDICES once, then apply those exact indices to X
        # and y. This prevents any possibility of misalignment caused by
        # independently splitting the arrays.
        all_idx = np.arange(len(X))
        # 60% of the training data fits the base TDF; 40% (context_holdout) is reserved for
        # the context step.
        base_idx, context_idx = train_test_split(
            all_idx,
            test_size=self.context_holdout,
            random_state=self.random_state,
            stratify=y if np.unique(y).size > 1 else None,
        )
        # Split that 40% in half: one half DISCOVERS regimes, the other half TUNES each
        # regime's rule.
        disc_local_idx, tune_local_idx = train_test_split(
            np.arange(len(context_idx)),
            test_size=0.5,
            random_state=self.random_state,
            stratify=y[context_idx] if np.unique(y[context_idx]).size > 1 else None,
        )
        disc_idx = context_idx[disc_local_idx]
        tune_idx = context_idx[tune_local_idx]
        # Verify the three parts cover all rows with no overlap.
        assert len(base_idx) + len(disc_idx) + len(tune_idx) == len(X)
        assert len(set(base_idx) | set(disc_idx) | set(tune_idx)) == len(X)
        assert set(base_idx).isdisjoint(set(disc_idx)) and set(base_idx).isdisjoint(set(tune_idx))
        assert set(disc_idx).isdisjoint(set(tune_idx))

        base_X, base_y = X[base_idx], y[base_idx]
        X_disc, y_disc = X[disc_idx], y[disc_idx]
        X_tune, y_tune = X[tune_idx], y[tune_idx]

        # The shared global TDF, fitted on the base portion only.
        self.base = TrustDriftFusionModel(
            random_state=self.random_state,
            weight_step=self.weight_step,
            min_weight=self.min_weight,
            **self.base_kwargs,
        ).fit(base_X, base_y)

        # Base TDF components are frozen after fit. Context discovery/tuning
        # are performed on disjoint rows that were not used to fit those
        # detectors or their global weights/threshold.
        AS_disc, RDS_disc, RF_disc = self.base._components(X_disc)
        AS_tune, RDS_tune, RF_tune = self.base._components(X_tune)

        # Regimes are discovered with KMeans on the [AS, RDS, RF] signal space of the discovery
        # rows; the tuning rows are then assigned to the nearest regime centre.
        comp_disc = np.column_stack([AS_disc, RDS_disc, RF_disc])
        max_clusters = min(self.n_clusters, len(comp_disc))
        if max_clusters < 1:
            raise ValueError("No rows available for regime discovery.")
        # n_init=10 restarts KMeans from 10 starting points and keeps the best, for stability.
        self.kmeans = KMeans(
            n_clusters=max_clusters,
            random_state=self.random_state,
            n_init=10,
        )
        discovery_context = self.kmeans.fit_predict(comp_disc)
        tune_context = self.kmeans.predict(
            np.column_stack([AS_tune, RDS_tune, RF_tune])
        )
        self.regimes = sorted(set(discovery_context.tolist()))
        self.discovery_context_ = discovery_context
        self.tuning_context_ = tune_context

        # Per-regime parameters, filled in the loop below.
        self.regime_weights = {}
        self.regime_thresholds = {}

        for regime in self.regimes:
            # Discovery determines which regime exists; tuning determines its
            # weights and threshold. There is no reuse of the discovery rows
            # for decision-rule optimization.
            tune_mask = tune_context == regime
            n_pos = y_tune[tune_mask].sum()

            # Too few rows, or only one class, in this regime: tuning would be unreliable,
            # so fall back to the global weights and threshold.
            if tune_mask.sum() < 20 or n_pos == 0 or n_pos == tune_mask.sum():
                self.regime_weights[regime] = (
                    self.base.w_as,
                    self.base.w_rds,
                    self.base.w_rf,
                )
                self.regime_thresholds[regime] = self.base.threshold
                continue

            AS_r = AS_tune[tune_mask]
            RDS_r = RDS_tune[tune_mask]
            RF_r = RF_tune[tune_mask]
            y_r = y_tune[tune_mask]

            best_f1 = -1.0
            best_w = (self.base.w_as, self.base.w_rds, self.base.w_rf)
            best_t = self.base.threshold
            # Same grid search as the global model, but restricted to this regime's tuning
            # rows.
            grid = np.arange(0, 1 + 1e-9, self.weight_step)

            for w_as in grid:
                for w_rds in grid:
                    w_rf = 1 - w_as - w_rds
                    if w_rf < 0 or w_rf > 1:
                        continue
                    if self.min_weight > 0 and (
                        w_as < self.min_weight
                        or w_rds < self.min_weight
                        or w_rf < self.min_weight
                    ):
                        continue

                    HT = w_as * AS_r + w_rds * RDS_r + w_rf * RF_r
                    t = best_f1_threshold_local(y_r, HT)
                    pred = (HT >= t).astype(int)
                    f1 = f1_score(y_r, pred, zero_division=0)

                    if f1 > best_f1:
                        best_f1, best_w, best_t = f1, (w_as, w_rds, w_rf), t

            self.regime_weights[regime] = best_w
            self.regime_thresholds[regime] = best_t

        # Retain explicit source-row indices for auditability and reproducibility.
        self.base_indices_ = base_idx
        self.discovery_indices_ = disc_idx
        self.tuning_indices_ = tune_idx

        return self

    # Which regime does each row belong to? The nearest KMeans centre in [AS, RDS, RF] space.
    def _assign_regime(self, X):
        AS, RDS, RF = self.base._components(X)
        comp = np.column_stack([AS, RDS, RF])
        return self.kmeans.predict(comp)

    def _weights_for_regime(self, regime):
        return self.regime_weights.get(
            regime,
            (self.base.w_as, self.base.w_rds, self.base.w_rf),
        )

    def _threshold_for_regime(self, regime):
        return self.regime_thresholds.get(regime, self.base.threshold)

    # Fuse the signals using each row's regime-specific weights.
    def score(self, X):
        AS, RDS, RF = self.base._components(X)
        regime_ids = self._assign_regime(X)
        HT = np.zeros(len(X))

        for regime in np.unique(regime_ids):
            mask = regime_ids == regime
            if not np.any(mask):
                continue
            w_as, w_rds, w_rf = self._weights_for_regime(regime)
            HT[mask] = w_as * AS[mask] + w_rds * RDS[mask] + w_rf * RF[mask]

        return np.clip(HT, 0, 1)

    # Apply each regime's own threshold.
    def predict(self, X):
        regime_ids = self._assign_regime(X)
        scores = self.score(X)
        pred = np.zeros(len(X), dtype=int)

        for regime in np.unique(regime_ids):
            mask = regime_ids == regime
            pred[mask] = (
                scores[mask] >= self._threshold_for_regime(regime)
            ).astype(int)

        return pred

    def summary(self):
        lines = [
            f"Context-Aware TDF (clustered regimes, {len(self.regimes)} discovered):",
            "  Regime discovery and regime tuning use disjoint held-out subsets.",
        ]
        for r in self.regimes:
            w_as, w_rds, w_rf = self._weights_for_regime(r)
            lines.append(
                f"  Regime {r}: w_as={w_as:.2f} w_rds={w_rds:.2f} "
                f"w_rf={w_rf:.2f} threshold={self._threshold_for_regime(r):.2f}"
            )
        return "\n".join(lines)


### What this cell does: run the context-aware model
Fits the context-aware model on training data, then compares it with the global TDF from Table 1 on the test set.

In [ ]:
# ==============================================================================
# RUN + COMPARE the context-aware model against the global TDF on the held-out test set.
# ==============================================================================
print("Fitting Context-Aware TDF (3 auto-discovered regimes; disjoint discovery/tuning)...")
# 3 regimes discovered by KMeans; only training data is passed to fit().
ctx_model = ContextAwareTDF(n_clusters=3, random_state=RANDOM_STATE)
ctx_model.fit(X_train, y_train)
print(ctx_model.summary())

# First contact with the test set for this model: scores and predictions.
ctx_scores = ctx_model.score(X_test)
ctx_pred = ctx_model.predict(X_test)

# Global TDF numbers come from Table 1 (`results`) for a side-by-side comparison.
print("\n=== Global (single-weight) TDF ===")
print(f"  F1: {results['TDF (Proposed)']['F1-Score']:.2f}  MCC: {results['TDF (Proposed)']['MCC']:.2f}")
print("\n=== Context-Aware TDF ===")
print(f"  Accuracy:  {accuracy_score(y_test, ctx_pred)*100:.2f}")
print(f"  F1-Score:  {f1_score(y_test, ctx_pred, zero_division=0)*100:.2f}")
print(f"  MCC:       {matthews_corrcoef(y_test, ctx_pred)*100:.2f}")
print(f"  Bal.Acc:   {balanced_accuracy_score(y_test, ctx_pred)*100:.2f}")
print(f"  ROC-AUC:    {roc_auc_score(y_test, ctx_scores)*100:.2f}")
print(f"  PR-AUC:     {average_precision_score(y_test, ctx_scores)*100:.2f}")


## 6. Cost-sensitive severity thresholds
Instead of fixed 80/60/35 Trust Score cutoffs, derive them from an explicit cost
model: how many times worse is a missed attack than an unnecessary quarantine?

### What this cell does: cost-sensitive thresholds
In security a missed attack usually costs more than a false alarm. Here the threshold minimises `5*FN + 1*FP`, and the severity-band cutoffs are derived from cost ratios (1x / 3x / 8x) instead of the fixed 80 / 60 / 35. Expect higher recall and lower precision than the F1-optimal model.

In [ ]:
# ==============================================================================
# COST-SENSITIVE THRESHOLDS (Section 6). A missed attack is usually worse than a false
# alarm. Choose the threshold by minimising a cost instead of maximising F1, and derive
# severity-band cutoffs from cost ratios.
# ==============================================================================
print("=== F1-optimal threshold (default) ===")
print(f"  threshold={tdf_model.threshold:.2f}  weights: w_as={tdf_model.w_as:.2f} "
      f"w_rds={tdf_model.w_rds:.2f} w_rf={tdf_model.w_rf:.2f}")

print("\n=== Cost-sensitive threshold (missed attack = 5x worse than false quarantine) ===")
# Same model, but the threshold minimises 5*FN + 1*FP, which favours higher recall at the
# price of lower precision.
cost_model = TrustDriftFusionModel(random_state=RANDOM_STATE, threshold_mode="cost", cost_fn=5.0, cost_fp=1.0)
cost_model.fit(X_train, y_train)
print(f"  threshold={cost_model.threshold:.2f}  weights: w_as={cost_model.w_as:.2f} "
      f"w_rds={cost_model.w_rds:.2f} w_rf={cost_model.w_rf:.2f}")

# Test-set evaluation of the cost-tuned model, compared with the F1-optimal TDF.
cost_scores = cost_model.score(X_test)
cost_pred = (cost_scores >= cost_model.threshold).astype(int)
print(f"\n  Test-set Recall: {recall_score(y_test, cost_pred)*100:.2f}%  "
      f"(vs {results['TDF (Proposed)']['Recall']:.2f}% for the F1-optimal version)")
print(f"  Test-set Precision: {precision_score(y_test, cost_pred, zero_division=0)*100:.2f}%  "
      f"(vs {results['TDF (Proposed)']['Precision']:.2f}%)")

print("\n=== Cost-sensitive severity bands (schedule: 1x / 3x / 8x) ===")
# Cutoffs for Trusted / Monitored / Elevated / Critical derived from FN:FP cost ratios 1x /
# 3x / 8x, instead of the fixed 80 / 60 / 35.
bands = severity_bands_from_cost(cost_model, cost_schedule=(1.0, 3.0, 8.0))
print(f"  Trusted >= {bands[0]:.1f}  |  Monitored >= {bands[1]:.1f}  |  "
      f"Elevated >= {bands[2]:.1f}  |  else Critical")
print("  (compare to the paper's fixed 80 / 60 / 35 cutoffs)")


## 7. Explainability: real SHAP attributions
Replaces a templated explanation with actual computed SHAP feature attribution on
the Random Forest component, matching the paper's 4-field explanation structure
(risk_summary, evidence, feature_attribution, recommended_action).

### What this cell does: explainability (SHAP)
Defines `TDFExplainer`. For one network window it recomputes AS, RDS and RF, fuses them, converts the result to a Trust Score and severity band, and uses SHAP to name the features that pushed the Random Forest toward or away from "Attack". Output has four fields: `risk_summary`, `evidence`, `feature_attribution`, `recommended_action`.

In [ ]:
# ==============================================================================
# EXPLAINABILITY (Section 7). Wraps a fitted TDF so any single row gets a 4-field
# explanation (risk_summary, evidence, feature_attribution, recommended_action) using real
# SHAP values from the Random Forest component.
# ==============================================================================
"""
Explainability Generation (Algorithm Step 8 / Eq. E in the paper),
implemented for real using SHAP feature attribution on the Random Forest
component, instead of a templated placeholder.

Closes the exact gap the paper's own related work identifies: "SHAP...
generally used as an additional post-hoc audit layer for a single
classifier output" -- here it is wired directly into the live fused
scoring pipeline, and its output feeds the same 4-field structure
(risk_summary, evidence, feature_attribution, recommended_action)
described in the architecture.

Usage:
    model = TrustDriftFusionModel().fit(X_train, y_train)
    explainer = TDFExplainer(model, feature_names)
    explanation = explainer.explain(X_val[0])
    print(explanation)
"""
import shap


# (minimum Trust Score, band name, recommended action), highest first. These are the paper's
# fixed 80 / 60 / 35 cutoffs.
SEVERITY_ACTIONS = [
    (80, "Trusted", "Monitor"),
    (60, "Monitored", "Log & Watch"),
    (35, "Elevated Risk", "Isolate"),
    (0, "Critical", "Quarantine & Escalate"),
]


# Map a Trust Score to (band, action): the first cutoff the score reaches decides the band.
def trust_score_to_band_with_actions(trust_score, severity_actions):
    """Map a trust score using caller-supplied bands."""
    for cutoff, band, action in severity_actions:
        if trust_score >= cutoff:
            return band, action
    return severity_actions[-1][1], severity_actions[-1][2]


# Wraps a fitted TDF with a SHAP TreeExplainer over its Random Forest.
class TDFExplainer:
    """Wraps a fitted TrustDriftFusionModel with a SHAP TreeExplainer over
    its Random Forest component, producing real per-window explanations."""

    def __init__(self, model, feature_names, top_k=3, severity_actions=None):
        self.model = model
        self.feature_names = list(feature_names)
        self.top_k = top_k
        self.severity_actions = (
            list(severity_actions) if severity_actions is not None
            else list(SEVERITY_ACTIONS)
        )
        # TreeExplainer on the RF component -- exact (not approximate) SHAP
        # values for tree ensembles, no background sampling needed for the
        # standard tree_path_dependent perturbation.
        self.shap_explainer = shap.TreeExplainer(model.rf)

    def _rf_shap_values(self, x_row):
        # The RF was trained on standardised data, so apply the same scaler first.
        Xs = self.model.scaler.transform(x_row.reshape(1, -1))
        sv = self.shap_explainer.shap_values(Xs)
        # shap_values may return a list (per-class) or a single array
        # depending on the sklearn/shap version; normalize to the
        # positive-class (Attack) contributions.
        if isinstance(sv, list):
            contrib = sv[1][0]
        elif sv.ndim == 3:
            contrib = sv[0, :, 1]
        else:
            contrib = sv[0]
        return contrib

    def explain(self, x_row, prev_trust=100.0, alpha=0.3):
        """Produces the 4-field explanation for a single telemetry window,
        computed from the model's actual fused score and real SHAP
        attributions -- not a template."""
        x_row = np.asarray(x_row, dtype=float)
        # Recompute the three signals for this one row and fuse them into HT.
        AS, RDS, RF = self.model._components(x_row.reshape(1, -1))
        AS, RDS, RF = float(AS[0]), float(RDS[0]), float(RF[0])
        HT = self.model.w_as * AS + self.model.w_rds * RDS + self.model.w_rf * RF
        HT = float(np.clip(HT, 0, 1))

        # Trust Score = 100 - 100*HT, smoothed with the previous score: T = alpha*(100 -
        # 100*HT) + (1 - alpha)*previous. With the defaults (previous = 100, alpha = 0.3) a
        # single window lowers trust only gradually.
        trust_score = alpha * (100 - 100 * HT) + (1 - alpha) * prev_trust
        band, action = trust_score_to_band_with_actions(trust_score, self.severity_actions)

        # Real SHAP feature attribution for the RF component
        contrib = self._rf_shap_values(x_row)
        # Top-k features by absolute SHAP value. A positive SHAP value pushes the prediction
        # toward Attack.
        order = np.argsort(np.abs(contrib))[::-1][: self.top_k]
        feature_attribution = [
            {"feature": self.feature_names[i],
             "shap_value": round(float(contrib[i]), 4),
             "direction": "increases risk" if contrib[i] > 0 else "decreases risk"}
            for i in order
        ]

        # Human-readable evidence: the three raw signals behind the decision.
        evidence = [
            f"Anomaly score {AS:.2f} (Isolation Forest, unsupervised)",
            f"Resource drift score {RDS:.2f} (training-reference Z-score signal)",
            f"Attack probability {RF:.2f} (Random Forest, w={self.model.w_rf:.2f})",
        ]

        top_feat = feature_attribution[0]["feature"] if feature_attribution else "n/a"
        risk_summary = (
            f"{band}: fused threat score {HT:.2f} (Trust Score {trust_score:.1f}). "
            f"Most influential feature: {top_feat}."
        )

        # The 4 explanation fields plus a few numeric extras.
        return {
            "risk_summary": risk_summary,
            "evidence": evidence,
            "feature_attribution": feature_attribution,
            "recommended_action": action,
            "trust_score": round(trust_score, 2),
            "severity_band": band,
            "hybrid_threat_score": round(HT, 4),
        }


### What this cell does: explanation demo
Prints explanations for the three highest-risk attack windows from the *validation* partition (the test set is kept out of demos).

In [ ]:
# ==============================================================================
# DEMO: explain the 3 highest-risk Attack windows from the validation partition (not the
# test set, which stays reserved for benchmarking).
# ==============================================================================
explainer = TDFExplainer(tdf_model, feature_cols)

# Test data are reserved for final benchmark evaluation. For illustrative SHAP
# examples, select representative high-risk Attack windows from TDF's
# non-test validation partition instead.
attack_idx = np.where(tdf_model.y_val_ == 1)[0]
val_scores = tdf_model.score(tdf_model.X_val_)
# Sort the attack windows from highest to lowest fused risk score.
ranked = attack_idx[np.argsort(val_scores[attack_idx])[::-1]]

print("Explaining the top 3 highest-risk Attack windows from the TDF validation partition:\n")
for i in ranked[:3]:
    exp = explainer.explain(tdf_model.X_val_[i])
    print("=" * 70)
    print(f"risk_summary:       {exp['risk_summary']}")
    print(f"severity_band:      {exp['severity_band']}")
    print(f"recommended_action: {exp['recommended_action']}")
    print("evidence:")
    for e in exp["evidence"]:
        print(f"  - {e}")
    print("feature_attribution:")
    for f in exp["feature_attribution"]:
        print(f"  - {f['feature']}: SHAP={f['shap_value']} ({f['direction']})")
    print()


## 8. Drift-adaptive retraining
The base TDF computes RDS but never *acts* on it. This closes the loop: track a
rolling EMA of the RDS signal, and auto-retrain the detectors when sustained drift is
detected. Demonstrated on a synthetic stream built by concatenating three
different device-profile blocks in sequence (sensor -> gateway -> actuator),
which produces a genuine, sharp regime shift at each device boundary - a
realistic scenario (e.g. a fleet composition change) rather than a purely
gradual drift. **Note:** retraining is triggered correctly, but recovery is
not always instantaneous in the very next batch - report this honestly
rather than claiming a clean win every time; the comparison below shows
real, unedited numbers.

**Protocol note (retraining-threshold leakage control):** `TrustDriftFusionModel.fit()` carves its own sub-train/validation split out of whatever buffer it is given and freezes the threshold and fusion weights on the validation rows only. `DriftAdaptiveTDF` reuses that frozen threshold for the initial fit and for every retrain, so no threshold is ever selected on rows the detectors were fit on.


### What this cell does: drift-adaptive retraining (class definition)
`DriftAdaptiveTDF` processes data in batches. For each batch it (1) scores with the current model, (2) updates a smoothed drift value (EMA of mean RDS), and (3) retrains on a rolling buffer when drift stays high for several consecutive batches. Because retraining uses labels from the buffer, this is *label-assisted* adaptation.

In [ ]:
# ==============================================================================
# DRIFT-ADAPTIVE RETRAINING (Section 8). Processes a labelled stream in batches, tracks an
# exponential moving average (EMA) of the drift score RDS, and refits the model on a rolling
# buffer when drift is sustained. It is label-assisted: retraining uses true labels from the
# buffer.
# ==============================================================================
"""
Drift-triggered, label-assisted adaptive retraining for Trust-Drift Fusion.

The base TrustDriftFusionModel computes a Resource Drift Score
(RDS) but never *acts* on it — it's just one of three fused inputs. This
module closes that loop: DriftAdaptiveTDF processes a labeled stream in
batches, tracks an EMA of the mean RDS over recent batches, and when that
EMA crosses a threshold for a sustained number of batches, it automatically
refits the Isolation Forest, Random Forest, and fusion weights on the most
recent data — without any human retraining trigger.

This directly operationalizes the "drift signal not fed back into a
downstream decision" gap identified in the paper's own related-work
section (refs [6]-[8]), and produces a concrete before/after comparison:
performance over time WITH drift-triggered retraining vs. a frozen model.

Usage:
    model = DriftAdaptiveTDF(base_kwargs=dict(n_estimators=200))
    history = model.stream_evaluate(X_stream, y_stream, batch_size=200)
    # history is a list of dicts: batch index, metrics, whether a retrain
    # fired on that batch, and the rolling drift EMA value.
"""


# Streaming loop: score each batch with the CURRENT model -> update the drift EMA -> if the
# EMA stays above the threshold for N consecutive batches -> retrain on the rolling buffer.
class DriftAdaptiveTDF:
    """Wraps TrustDriftFusionModel with a rolling buffer + drift-EMA trigger
    that automatically refits the underlying model when sustained resource
    drift is detected in the incoming stream."""

    def __init__(self, base_kwargs=None, buffer_size=1000,
                 drift_ema_alpha=0.2, drift_threshold=0.35,
                 sustained_batches=3, min_buffer_for_retrain=200):
        self.base_kwargs = base_kwargs or {}
        self.buffer_size = buffer_size
        self.drift_ema_alpha = drift_ema_alpha
        self.drift_threshold = drift_threshold
        self.sustained_batches = sustained_batches
        self.min_buffer_for_retrain = min_buffer_for_retrain

        self.model = None
        self.threshold = 0.5
        self.buffer_X = []
        self.buffer_y = []
        # drift_ema = smoothed mean RDS; _breaches_in_a_row counts consecutive batches above
        # the threshold.
        self.drift_ema = 0.0
        self._breaches_in_a_row = 0
        self.retrain_events = []  # batch indices where a retrain fired

    def _initial_fit(self, X0, y0):
        # TrustDriftFusionModel.fit() already carves its own sub-train/
        # validation split out of (X0, y0) internally, and freezes
        # self.threshold (and the fusion weights) using ONLY that held-out
        # validation partition -- X0 itself is never touched again after
        # that split is made. Correction: this method used to re-derive a
        # threshold via best_f1_threshold_local(y0, HT) on
        # self.model._components(X0), i.e. scored on the SAME rows (X0)
        # used to fit the Isolation Forest / Random Forest / drift
        # baseline -- selecting a threshold on data the model was fit on
        # rather than on a validation split. Fixed: just reuse the model's
        # own leakage-free threshold.
        self.model = TrustDriftFusionModel(**self.base_kwargs).fit(X0, y0)
        self.threshold = self.model.threshold
        # The buffer starts with the warm-up data.
        self.buffer_X = [X0]
        self.buffer_y = [y0]

    def _retrain(self, batch_idx):
        X_buf = np.concatenate(self.buffer_X, axis=0)
        y_buf = np.concatenate(self.buffer_y, axis=0)
        if len(np.unique(y_buf)) < 2 or len(y_buf) < self.min_buffer_for_retrain:
            return  # not enough signal to safely retrain yet
        # Same fix as _initial_fit: TrustDriftFusionModel.fit() carves a
        # sub-train/validation split out of (X_buf, y_buf) and freezes
        # self.threshold using only the held-out validation rows of THIS
        # retraining buffer. Reuse that threshold directly rather than
        # recomputing one from components scored on the full X_buf (which
        # includes the rows just used to fit the refreshed detectors).
        self.model = TrustDriftFusionModel(**self.base_kwargs).fit(X_buf, y_buf)
        self.threshold = self.model.threshold
        self.retrain_events.append(batch_idx)
        # Reset the drift state after retraining so it does not immediately re-trigger.
        self.drift_ema = 0.0
        self._breaches_in_a_row = 0

    def _metrics(self, y_true, scores, pred):
        return {
            "Accuracy": accuracy_score(y_true, pred) * 100,
            "Precision": precision_score(y_true, pred, zero_division=0) * 100,
            "Recall": recall_score(y_true, pred, zero_division=0) * 100,
            "F1-Score": f1_score(y_true, pred, zero_division=0) * 100,
            "ROC-AUC": (roc_auc_score(y_true, scores) * 100
                        if len(np.unique(y_true)) > 1 else float("nan")),
            "PR-AUC": (average_precision_score(y_true, scores) * 100
                       if len(np.unique(y_true)) > 1 else float("nan")),
            "MCC": matthews_corrcoef(y_true, pred) * 100,
            "Balanced Accuracy": balanced_accuracy_score(y_true, pred) * 100,
        }

    def stream_evaluate(self, X_stream, y_stream, batch_size=200,
                         warmup_batches=1, enable_retrain=True):
        """Processes X_stream/y_stream in sequential batches, scoring each
        batch with the CURRENT model before any retraining on that batch's
        own data (so metrics reflect real streaming performance, not
        hindsight), then updates the rolling buffer and checks the drift
        trigger. Returns a list of per-batch result dicts."""
        n = len(X_stream)
        n_batches = int(np.ceil(n / batch_size))
        history = []

        # Warm-start: fit on the first `warmup_batches` batches.
        # Warm start: the first batch(es) train the initial model.
        warm_end = warmup_batches * batch_size
        self._initial_fit(X_stream[:warm_end], y_stream[:warm_end])

        for b in range(warmup_batches, n_batches):
            lo, hi = b * batch_size, min((b + 1) * batch_size, n)
            Xb, yb = X_stream[lo:hi], y_stream[lo:hi]
            if len(Xb) == 0:
                continue

            # Score BEFORE any retraining on this batch, so the metrics reflect real
            # streaming performance (no hindsight).
            scores = self.model.score(Xb)
            pred = (scores >= self.threshold).astype(int)
            metrics = self._metrics(yb, scores, pred)

            # Drift signal for this batch = mean RDS of its rows.
            _, RDS_b, _ = self.model._components(Xb)
            batch_rds = float(np.mean(RDS_b))
            # EMA update: ema = alpha*batch_rds + (1 - alpha)*ema. Smooths out one-off
            # spikes.
            self.drift_ema = (self.drift_ema_alpha * batch_rds
                               + (1 - self.drift_ema_alpha) * self.drift_ema)

            retrained = False
            # Trigger rule: retrain only after `sustained_batches` consecutive breaches.
            # enable_retrain=False gives the 'frozen model' control run.
            if enable_retrain:
                if self.drift_ema >= self.drift_threshold:
                    self._breaches_in_a_row += 1
                else:
                    self._breaches_in_a_row = 0
                if self._breaches_in_a_row >= self.sustained_batches:
                    self._retrain(b)
                    retrained = True

            # update rolling buffer (fixed-size FIFO over rows, not batches)
            self.buffer_X.append(Xb)
            self.buffer_y.append(yb)
            total = sum(len(x) for x in self.buffer_X)
            while total > self.buffer_size and len(self.buffer_X) > 1:
                total -= len(self.buffer_X[0])
                self.buffer_X.pop(0)
                self.buffer_y.pop(0)

            history.append({
                "batch": b, "n": len(Xb), "drift_ema": self.drift_ema,
                "retrained": retrained, **metrics,
            })

        return history


### What this cell does: build a synthetic drifting stream
Generates 3,000 synthetic windows (10 invented features, **not** TON_IoT) from three device profiles in sequence (sensor, gateway, actuator), with 18% attacks. The sudden change between profiles is a genuine regime shift that tests the drift logic.

In [ ]:
# ==============================================================================
# SYNTHETIC REGIME-SHIFT STREAM. Entirely synthetic (10 made-up telemetry features, NOT
# TON_IoT). Three device profiles are concatenated so the data changes sharply at each
# boundary, which is what the drift experiment needs.
# ==============================================================================
# --- Build a synthetic stream with a genuine REGIME SHIFT: three
# device-profile blocks concatenated in sequence (sensor -> gateway ->
# actuator). This produces a sharp, real distribution change at each
# boundary, unlike a randomly-shuffled dataset which has no true "time".
# Feature names (documentation only; the code below addresses columns by index 0-9 in this
# order).
FEATURES_SYNTH = [
    "connection_rate", "unique_peer_count", "protocol_diversity",
    "byte_volume_ratio", "packet_count_ratio",
    "session_duration_mean", "session_duration_var",
    "cpu_utilization", "memory_utilization", "error_rate",
]
# Seven attack styles; each one distorts a few specific features.
ATTACK_TYPES = ["reconnaissance", "credential_stuffing", "c2_beaconing",
                 "resource_flooding", "protocol_anomaly", "slow_exfiltration", "firmware_tamper"]

# Normal behaviour for one device profile: a mean and std-dev per feature.
def make_baseline(rng, n, profile):
    base = {
        "sensor": dict(mean=[5,3,1.2,0.4,0.4,40,8,12,18,0.01], std=[1.5,1,0.3,0.1,0.1,8,3,4,5,0.01]),
        "gateway": dict(mean=[40,15,3.5,0.9,0.9,20,10,35,45,0.03], std=[8,4,0.8,0.2,0.2,6,4,8,10,0.02]),
        "actuator": dict(mean=[8,4,1.5,0.3,0.3,15,5,15,22,0.02], std=[2,1,0.4,0.1,0.1,4,2,4,6,0.01]),
    }[profile]
    # A slow random walk added to simulate gradual drift inside a profile.
    drift = rng.normal(0, 0.02, size=(n, 10)).cumsum(axis=0)
    Xb = rng.normal(base["mean"], base["std"], size=(n, 10)) + drift * np.array(base["std"])
    return np.clip(Xb, 0, None)

# Turn a normal row into an attack row by scaling/shifting selected features. Indices: 0
# connection_rate, 1 unique_peer_count, 2 protocol_diversity, 3 byte_volume_ratio, 4
# packet_count_ratio, 5 session_duration_mean, 6 session_duration_var, 7 cpu_utilization, 8
# memory_utilization, 9 error_rate.
def inject_attack(row, attack_type, rng):
    row = row.copy()
    if attack_type == "reconnaissance": row[0]*=rng.uniform(3,6); row[1]*=rng.uniform(2,4)
    elif attack_type == "credential_stuffing": row[0]*=rng.uniform(4,8); row[9]+=rng.uniform(0.2,0.4)
    elif attack_type == "c2_beaconing": row[3]*=rng.uniform(1.5,2.5); row[5]*=rng.uniform(2,3)
    elif attack_type == "resource_flooding": row[7]+=rng.uniform(30,55); row[0]*=rng.uniform(5,10)
    elif attack_type == "protocol_anomaly": row[2]*=rng.uniform(2,4)
    elif attack_type == "slow_exfiltration": row[3]*=rng.uniform(1.1,1.3); row[4]*=rng.uniform(1.1,1.3)
    elif attack_type == "firmware_tamper": row[8]+=rng.uniform(20,40); row[9]+=rng.uniform(0.1,0.3)
    return row

# Build the stream: 1000 windows per profile in the order sensor -> gateway -> actuator;
# each window is an attack with probability 18%. The profile switch is the regime shift.
rng = np.random.default_rng(RANDOM_STATE)
n_per_device = 1000
X_stream, y_stream = [], []
for profile in ["sensor", "gateway", "actuator"]:
    Xb = make_baseline(rng, n_per_device, profile)
    for w in range(n_per_device):
        label, feat = 0, Xb[w]
        if rng.random() < 0.18:
            feat = inject_attack(feat, rng.choice(ATTACK_TYPES), rng); label = 1
        X_stream.append(feat); y_stream.append(label)
X_stream, y_stream = np.array(X_stream), np.array(y_stream)
print(f"Generated regime-shift stream: {len(X_stream)} windows "
      f"(sensor 0-999, gateway 1000-1999, actuator 2000-2999), {y_stream.sum()} attack")


### What this cell does: run retraining ON vs OFF
Runs the same stream through the adaptive model and a frozen control, prints which batches triggered retraining and the per-batch accuracy of both, then the next cell plots them.

In [ ]:
# ==============================================================================
# A/B RUN of the drift experiment: the same stream processed by the same class with
# retraining ON vs OFF (frozen model).
# ==============================================================================
print("Drift-adaptive experiment: LABEL-ASSISTED retraining")
print("Running WITH drift-triggered retraining...")
# drift_threshold=0.3: retrain when the smoothed drift stays above 0.3 for the required
# number of batches.
adaptive = DriftAdaptiveTDF(drift_threshold=0.3)
hist_adaptive = adaptive.stream_evaluate(X_stream, y_stream, batch_size=300, enable_retrain=True)

print("Running WITHOUT retraining (frozen model)...")
# Control: identical settings but retraining disabled.
frozen = DriftAdaptiveTDF(drift_threshold=0.3)
hist_frozen = frozen.stream_evaluate(X_stream, y_stream, batch_size=300, enable_retrain=False)

# Which batches triggered a retrain.
print(f"\nRetrain events fired at batches: {adaptive.retrain_events}\n")
print(f"{'batch':>5} {'drift_ema':>10} {'retrained':>10} | {'Acc(adaptive)':>14} {'Acc(frozen)':>12}")
# Per-batch accuracy of both runs, side by side.
for a, f in zip(hist_adaptive, hist_frozen):
    print(f"{a['batch']:5d} {a['drift_ema']:10.3f} {'YES' if a['retrained'] else '-':>10} | "
          f"{a['Accuracy']:14.2f} {f['Accuracy']:12.2f}")


In [ ]:
# ==============================================================================
# FIGURE: accuracy over the stream, with and without drift-adaptive retraining.
# ==============================================================================
fig, ax = plt.subplots(figsize=(7, 3.2))
batches = [h["batch"] for h in hist_adaptive]
ax.plot(batches, [h["Accuracy"] for h in hist_adaptive], label="With drift-adaptive retraining", linewidth=1.8, color="#15803d")
ax.plot(batches, [h["Accuracy"] for h in hist_frozen], label="Frozen (no retraining)", linewidth=1.8, color="#b91c1c")
# Dashed vertical lines mark the batches where a retrain fired.
for b in adaptive.retrain_events:
    ax.axvline(b, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.set_xlabel("Batch"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Drift-adaptive retraining vs. frozen model over a drifting stream")
ax.legend(fontsize=8)
plt.tight_layout();
plt.savefig("drift_adaptive_comparison.pdf", dpi=600);
plt.savefig("drift_adaptive_comparison.png", dpi=600);
plt.show()


## 9. Adversarial robustness - controlled robustness stress test
This is a controlled diagnostic stress test, separate from the main held-out
test benchmark. It uses non-test validation attacks as the source cohort and
crafts feature-space evasions targeting the Random Forest component. No
threshold, fusion weight, hyperparameter, feature, or model selection is
performed here, and the final held-out test set remains reserved for final
evaluation.


### What this cell does: adversarial stress test
A simple attacker edits attack rows that TDF currently catches, moving the most important features toward the Normal average until the Random Forest is fooled. We then measure how often the RF alone, versus the fused TDF, is evaded. This is a limited diagnostic (white-box, RF-only attacker), not proof of robustness.

In [ ]:
# ==============================================================================
# ADVERSARIAL ROBUSTNESS (Section 9), a diagnostic stress test: a simple white-box attacker
# edits detected attack rows to look more normal, and we measure how often the Random Forest
# alone vs the fused TDF is fooled. Nothing is tuned here and the test set is not used.
# ==============================================================================
# Greedy evasion: for each attack row, nudge features toward the Normal mean until the RF no
# longer flags it (or max_steps is reached).
def craft_evasive_samples(model, X_attack, normal_mean, top_k=8, max_steps=25,
                            step_frac=0.15, rf_threshold=0.5):
    # The attacker is assumed to know which features matter most: the top_k by RF
    # importance.
    importances = model.rf.feature_importances_
    top_idx = np.argsort(importances)[::-1][:top_k]
    X_adv = X_attack.copy()

    for row in range(len(X_adv)):
        x = X_adv[row].copy()
        # Repeat: if the RF still says Attack (probability >= 0.5), move each top feature
        # 15% of the way toward the Normal mean; stop as soon as the RF is fooled.
        for _ in range(max_steps):
            Xs = model.scaler.transform(x.reshape(1, -1))
            rf_prob = model.rf.predict_proba(Xs)[0, 1]
            if rf_prob < rf_threshold:
                break
            for j in top_idx:
                x[j] = x[j] + step_frac * (normal_mean[j] - x[j])
        X_adv[row] = x

    return X_adv


# Robustness stress test is kept separate from the final held-out test benchmark.
# Use TDF's non-test validation partition as the attack source so the final test
# set remains reserved for final evaluation only.
# Average Normal row: the target the attacker moves toward.
normal_mean = X_train[y_train == 0].mean(axis=0)
attack_idx_val = np.where(tdf_model.y_val_ == 1)[0]
val_attack_scores = tdf_model.score(tdf_model.X_val_)
# Only attacks that TDF currently detects, so 'evasion success' means a real flip from
# detected to missed.
caught_val = attack_idx_val[val_attack_scores[attack_idx_val] >= tdf_model.threshold]

# Sample up to 150 of them.
adv_rng = np.random.default_rng(RANDOM_STATE)
n_craft = min(150, len(caught_val))
target_idx = adv_rng.choice(caught_val, size=n_craft, replace=False) if n_craft else np.array([], dtype=int)
X_attack = tdf_model.X_val_[target_idx]

print(f"Crafting evasions for {n_craft} validation Attack samples originally caught by TDF...")

X_adv = craft_evasive_samples(tdf_model, X_attack, normal_mean) if n_craft else X_attack

# RF probability alone, to compare with the fused TDF score.
def rf_alone_score(Xb):
    Xs = tdf_model.scaler.transform(Xb)
    return tdf_model.rf.predict_proba(Xs)[:, 1]

if n_craft:
    rf_after = rf_alone_score(X_adv)
    fused_after = tdf_model.score(X_adv)
    # Evasion success = % of crafted rows now scored below the decision boundary. The attack
    # only optimises the RF signal, so the AS and RDS signals in the fusion are not directly
    # attacked.
    rf_evasion = np.mean(rf_after < 0.5) * 100
    fused_evasion = np.mean(fused_after < tdf_model.threshold) * 100

    print(f"\nRF-alone evasion success:  {rf_evasion:.1f}%")
    print(f"TDF fused evasion success: {fused_evasion:.1f}%")
else:
    print("\nNo validation Attack samples were available for the diagnostic stress test.")
print("\nThis diagnostic does not alter any benchmark threshold, weight, hyperparameter,")
print("feature-selection decision, or held-out test result.")


## 10. Reviewer diagnostic: feature-group sensitivity / memorization check

This validation-only diagnostic addresses a reviewer concern about whether very high performance
depends disproportionately on DNS/HTTP/SSL fields or port numbers. It does **not** change the
principal TDF benchmark, its reported metrics, selected weights, thresholds, or test evaluation.

Each feature group is removed from the encoded matrix and a fresh Random Forest is trained on a
sub-training portion of the original training partition. Performance is measured on a separate
diagnostic validation portion. The final held-out test partition is not used by this diagnostic.


### What this cell does: feature-group sensitivity (reviewer diagnostic)
Checks whether very high scores rely on port numbers or DNS/HTTP/SSL fields. For each group, retrain a Random Forest without those columns and compare validation scores. Test data is not used.

In [ ]:
# ==============================================================================
# FEATURE-GROUP SENSITIVITY CHECK (reviewer diagnostic). Does the model's performance depend
# heavily on port numbers or on DNS / HTTP / SSL fields? Remove each group, retrain a Random
# Forest, and compare validation scores. Uses training data only; the test set is not
# touched.
# ==============================================================================
# Validation-only feature-group sensitivity diagnostic.
# This cell deliberately does NOT access y_test, so the final held-out test partition
# remains reserved for the principal benchmark and its associated figures/metrics.

# Each group is a rule over ENCODED column names (one-hot columns keep their prefix, e.g.
# 'dns_query_xxx').
FEATURE_GROUPS = {
    "ports": lambda c: c == "src_port" or c == "dst_port",
    "dns": lambda c: c == "dns_query" or c.startswith("dns_"),
    "http": lambda c: c == "http_uri" or c.startswith("http_"),
    "ssl": lambda c: c.startswith("ssl_"),
}

# Deterministic diagnostic split carved only from the original training partition.
_diag_sub_idx, _diag_val_idx = train_test_split(
    np.arange(len(X_train)), test_size=VALIDATION_SIZE, random_state=RANDOM_STATE,
    stratify=y_train if y_train.sum() > 1 else None,
)
X_diag_sub, X_diag_val = X_train[_diag_sub_idx], X_train[_diag_val_idx]
y_diag_sub, y_diag_val = y_train[_diag_sub_idx], y_train[_diag_val_idx]

# Boolean keep-mask: False for columns in the group. Returns None if no column matches.
def _encoded_drop_mask(feature_names, predicate):
    keep = np.array([not predicate(str(c)) for c in feature_names], dtype=bool)
    if keep.all():
        return None
    return keep

print("=== FEATURE-GROUP SENSITIVITY DIAGNOSTIC (VALIDATION ONLY) ===")
print("The final held-out test partition is intentionally not used in this diagnostic.")
# For each group: retrain a fresh RF without it. If scores barely change, the model is not
# relying on that group (addresses the memorisation concern).
for group_name, predicate in FEATURE_GROUPS.items():
    mask = _encoded_drop_mask(feature_cols, predicate)
    dropped = 0 if mask is None else int((~mask).sum())
    if mask is None:
        print(f"{group_name:>6}: group absent from encoded feature names; skipped")
        continue
    Xtr = X_diag_sub[:, mask]
    Xva = X_diag_val[:, mask]
    diag_rf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, max_depth=RF_MAX_DEPTH,
        class_weight="balanced", n_jobs=-1
    )
    diag_rf.fit(Xtr, y_diag_sub)
    diag_scores = diag_rf.predict_proba(Xva)[:, 1]
    # Fixed 0.5 cut-off. This is a diagnostic, not a tuned model.
    diag_pred = (diag_scores >= 0.5).astype(int)
    print(
        f"{group_name:>6}: dropped={dropped:4d} | "
        f"Acc={accuracy_score(y_diag_val, diag_pred)*100:6.2f} | "
        f"Prec={precision_score(y_diag_val, diag_pred, zero_division=0)*100:6.2f} | "
        f"Rec={recall_score(y_diag_val, diag_pred, zero_division=0)*100:6.2f} | "
        f"F1={f1_score(y_diag_val, diag_pred, zero_division=0)*100:6.2f} | "
        f"AUC={roc_auc_score(y_diag_val, diag_scores):.5f}"
    )
print("End diagnostic. These validation-only values are not part of the manuscript unless independently reviewed and reported.")


## 11. Final certification and protocol audit


## Benchmark separability caveat

TON_IoT network-flow classification can be highly separable under a random stratified split,
so very high held-out scores do not by themselves establish deployment-level robustness.
The present evaluation therefore treats the benchmark metrics as evidence for this
experimental split, not as proof of chronological generalization across devices, sessions,
or future operating regimes. This is why the study also reports synthetic drift adaptation,
controlled adversarial stress testing, and explicit limitations around temporal validation.


## Publication protocol audit

This notebook is the **source-of-truth evaluation script** for the reported experiment.
The main benchmark uses a held-out test partition only for final evaluation; preprocessing
vocabulary, model fitting, fusion weights, thresholds, and other selection decisions are
determined without test labels. The adaptive experiment is synthetic and label-assisted,
and the adversarial experiment is a controlled diagnostic stress test.

**Execution requirement:** this file is intentionally distributed without stale execution
outputs. A fresh-kernel **Run All** is required before the notebook can be described as
execution-certified. The final certification cell is the submission gate.


## Final execution and certification checklist

For the authoritative final run:

1. Start a fresh Python/Jupyter/Colab kernel.
2. Run the notebook from top to bottom without skipping cells.
3. Confirm the baseline cell prints TDF ROC-AUC and Average Precision to five decimals.
4. Confirm Table 1, the ablation table, the confusion matrix, ROC/PR figures, and drift figure are generated by that same run.
5. Confirm the final certification cell ends with:

`STATUS: PASS — all final integrity checks passed.`

6. Run the download cell (Section 12) if you need the result files, then save the executed notebook and keep the CSV/PNG artifacts produced by that exact run together with it.

No result in this clean source notebook should be treated as a newly executed result until the above Run-All has been completed.


### What this cell does: final integrity audit
Fail-closed gate. It re-checks split integrity, TDF settings, the existence of every result file, and that `table1_results.csv` matches the in-memory results; then writes `tdf_current_run_manifest.json` (run statistics plus SHA-256 hashes of the artifacts). It prints `STATUS: PASS` only if everything passed. Run it last.

In [ ]:
# ==============================================================================
# FINAL CERTIFICATION / INTEGRITY AUDIT. A fail-closed gate: 'STATUS: PASS' is printed only
# if every assertion below succeeds, i.e. the current kernel produced all artifacts and the
# numbers are consistent.
# ==============================================================================
# FINAL CERTIFICATION / INTEGRITY AUDIT
# Fail-closed: PASS is printed only after the current kernel has generated the required
# benchmark artifacts and the in-memory results satisfy structural and numerical checks.
print("=== FINAL INTEGRITY AUDIT ===")
print("Protocol: fresh-kernel Run All -> train/validation tuning -> frozen configuration -> held-out test evaluation")

# Result files that must exist and be non-empty after this run.
required = [
    "table1_results.csv",
    "ablation_results.csv",
    "confusion_matrix.png",
    "roc_curve.png",
    "pr_curve.png",
    "drift_adaptive_comparison.png",
]

# Core structural invariants.
# Group 1: data-split invariants (no lost / duplicated / overlapping rows, finite features,
# binary labels).
assert len(df) == RAW_ROW_COUNT, "Raw/preprocessed row count changed."
assert len(train_df) + len(test_df) == RAW_ROW_COUNT, "Train/test rows do not conserve the source rows."
assert set(train_df.index).isdisjoint(set(test_df.index)), "Train/test overlap detected."
assert len(set(train_df.index) | set(test_df.index)) == RAW_ROW_COUNT, "Rows are missing from the train/test union."
assert X_train.shape[1] == X_test.shape[1] == len(feature_cols), "Train/test feature matrices disagree."
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "Non-finite feature values remain."
assert set(np.unique(y_train)).issubset({0, 1}) and set(np.unique(y_test)).issubset({0, 1}), "Labels are not binary."

# TDF configuration invariants.
# Group 2: TDF configuration invariants (weights are non-negative and sum to 1, threshold
# within [0, 1]).
assert "TDF (Proposed)" in scores_by_model, "TDF result is missing."
tdf_model = fitted_models["TDF (Proposed)"]
assert abs((tdf_model.w_as + tdf_model.w_rds + tdf_model.w_rf) - 1.0) < 1e-12, "TDF weights do not sum to one."
assert all(w >= 0 for w in (tdf_model.w_as, tdf_model.w_rds, tdf_model.w_rf)), "Negative TDF weight detected."
assert 0 <= tdf_model.threshold <= 1, "TDF threshold is outside [0,1]."

# Exact headline traceability from the current in-memory held-out test scores.
# Recompute the headline metrics from in-memory test scores so the printed 5-decimal values
# are traceable.
tdf_scores = scores_by_model["TDF (Proposed)"]
tdf_auc = roc_auc_score(y_test, tdf_scores)
tdf_ap = average_precision_score(y_test, tdf_scores)
print(f"TDF ROC-AUC:           {tdf_auc:.5f}")
print(f"TDF Average Precision: {tdf_ap:.5f}")
print(f"TDF weights: AS={tdf_model.w_as:.2f}, RDS={tdf_model.w_rds:.2f}, RF={tdf_model.w_rf:.2f}")
print(f"TDF threshold: {tdf_model.threshold:.2f}")

# Required artifacts must exist and be non-empty after the current run.
# Group 3: every required artifact exists and is non-empty.
for fname in required:
    p = Path(fname)
    assert p.is_file() and p.stat().st_size > 0, f"Missing or empty required artifact: {fname}"

# Table 1 must match current in-memory results to displayed precision.
# Cross-check: the saved Table 1 file must match the in-memory results (guards against
# stale files from an earlier run).
t2 = pd.read_csv("table1_results.csv", index_col=0)
for model_name, metric_dict in results.items():
    assert model_name in t2.index, f"Missing model in table1_results.csv: {model_name}"
    for metric_name, value in metric_dict.items():
        assert metric_name in t2.columns, f"Missing metric column: {metric_name}"
        file_value = float(t2.loc[model_name, metric_name])
        assert abs(file_value - float(value)) < 5e-2, f"Table 1 mismatch: {model_name}/{metric_name}"

# Recompute and record artifact hashes only after verifying the artifacts exist.
# Manifest of this run: dataset sizes, seed, TDF weights/threshold, headline metrics and the
# SHA-256 hash of each artifact. Lets a reviewer verify the files belong to this exact run.
manifest = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "raw_rows": int(RAW_ROW_COUNT),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "feature_columns": int(len(feature_cols)),
    "random_seed": RANDOM_STATE,
    "tdf_weights": [float(tdf_model.w_as), float(tdf_model.w_rds), float(tdf_model.w_rf)],
    "tdf_threshold": float(tdf_model.threshold),
    "tdf_roc_auc": float(tdf_auc),
    "tdf_average_precision": float(tdf_ap),
    "artifacts": {},
}
for fname in required:
    p = Path(fname)
    manifest["artifacts"][fname] = {
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        "bytes": p.stat().st_size,
    }
# Write the manifest to disk.
with open("tdf_current_run_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)

assert Path("tdf_current_run_manifest.json").is_file() and Path("tdf_current_run_manifest.json").stat().st_size > 0
print("Required artifacts: present and non-empty")
print("Table 1: matches current in-memory results to rounding tolerance")
print("Manifest: SHA-256 recorded for all six required artifacts")
# Only reached if every check above passed.
print("STATUS: PASS — all final integrity checks passed.")


## 12. Download all results


### What this cell does: collect result files
In Colab it downloads the result files; elsewhere it lists which files exist in the working directory. It runs after the certification cell, so the run manifest already exists.


In [ ]:
# ==============================================================================
# DOWNLOAD HELPER. In Google Colab it triggers browser downloads of the result files;
# elsewhere it only reports which files exist in the working directory.
# ==============================================================================
# Portable result-download helper: Colab exposes `google.colab.files`;
# outside Colab, keep the notebook running and report the files that were saved
# in the current working directory instead of raising an import error.
import os

# Files to collect. tdf_current_run_manifest.json is written by the certification cell, which
# runs before this cell.
_DOWNLOAD_FILES = [
    "table1_results.csv",
    "ablation_results.csv",
    "confusion_matrix.png",
    "roc_curve.png",
    "pr_curve.png",
    "drift_adaptive_comparison.png",
    "tdf_current_run_manifest.json",
]

# Detect whether we are running inside Colab.
try:
    from google.colab import files as _colab_files
except ImportError:
    _colab_files = None

if _colab_files is not None:
    for fname in _DOWNLOAD_FILES:
        try:
            _colab_files.download(fname)
        except Exception as e:
            print(f"Skipped {fname}: {e}")
else:
    print("Not running in Google Colab; automatic browser downloads are unavailable.")
    print("Result files remain available in the current working directory:")
    for fname in _DOWNLOAD_FILES:
        status = "present" if os.path.exists(fname) else "missing"
        print(f"  {fname}: {status}")
